# ***Industrial Vision SLM Benchmark***

## ***Notebook 01 — Évaluation du modèle initial***

**Modèle :** Qwen3-1.7B (`Qwen/Qwen3-1.7B`)

**Environnement :** Google Colab — NVIDIA T4

**Objectif général :**

Construire un benchmark reproductible du SLM préentraîné,
sans RAG ni fine-tuning, afin de disposer d'une référence
pour les futures expérimentations.

**Jeu d'évaluation :**
- 50 questions techniques OpenCV.
- 10 questions de développement (`dev`).
- 40 questions de test (`test`).
- 5 domaines de traitement d'images et de vision industrielle.

**Métriques étudiées :**
- Exactitude et complétude des réponses.
- Hallucinations et erreurs factuelles.
- Latence d'inférence.
- Débit de génération (tokens/s).
- Consommation de mémoire GPU.

## **1. Préparation de l'environnement et du dataset**

**Objectif :** préparer Google Colab et récupérer le jeu
d'évaluation figé à l'issue de la section 9.

**Étapes :**
1. Vérifier la disponibilité du GPU.
2. Synchroniser le dépôt GitHub.
3. Vérifier l'intégrité du dataset avec SHA-256.
4. Charger les questions et contrôler leur répartition.

**Critères de validation :**
- GPU CUDA disponible.
- Dépôt GitHub accessible.
- Empreintes SHA-256 conformes au manifeste.
- 10 questions `dev` et 40 questions `test`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import torch

# Configuration du repo
REPO_URL = "https://github.com/I-am-Axel-Helios/industrial-vision-slm-benchmark.git"
PROJECT_DIR = Path("/content/industrial-vision-slm-benchmark")

# Clonage du repo, ou du moins actualisation du clone existant
if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

# Verif de l'environnement GPU
print("\n=== Environnement GPU ===")
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[WARNING] No GPU detected.")

# Vérif du dataset d'évaluation gelé
print("\n=== Vérification du dataset gelé ===")

subprocess.run([sys.executable, str(PROJECT_DIR / "src/freeze_evaluation_dataset.py"), "verify"], check=True)

# Lecture du dataset
dataset_path = PROJECT_DIR / "data/evaluation/questions.jsonl"

records = [
    json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines() if line.strip()
]

dev_questions = [record for record in records if record["split"] == "dev"]
test_questions = [record for record in records if record["split"] == "test"]

# Vérif du nombre de questions attendu
assert len(dev_questions) == 10, f"Nombre de questions 'dev' incorrect : {len(dev_questions)}"
assert len(test_questions) == 40, f"Nombre de questions 'test' incorrect : {len(test_questions)}"

print("\n=== Résumé du dataset ===")
print(f"Nombre total de questions : {len(records)}")
print(f"Questions 'dev' : {len(dev_questions)}")
print(f"Questions 'test' : {len(test_questions)}")
print("\n[OK] Environnement et dataset prêts.")

## **2. Chargement du modèle initial**

**Objectif :** charger le modèle préentraîné Qwen3-1.7B sur le GPU NVIDIA T4 afin de préparer son évaluation (réutiliser la configuration d'inférence validée dans le notebook 00)

**Configuration :**
- Modèle : `Qwen/Qwen3-1.7B`
- Précision : FP16
- Matériel : NVIDIA T4
- Sans RAG
- Sans fine-tuning

**Référence :** chargement et premières inférences réalisés dans le notebook 00.

**Résultats attendus :**
- Tokenizer chargé.
- Modèle chargé sur GPU.
- Nombre de paramètres affiché.
- Consommation mémoire GPU mesurée.

**Méthode :**
1. Charger le tokenizer Hugging Face.
2. Charger Qwen3-1.7B en précision FP16.
3. Placer le modèle sur le GPU.
4. Activer le mode évaluation.
5. Mesurer la mémoire GPU occupée.

**Point de vigilance :** le chargement peut nécessiter un téléchargement si les poids du modèle ne sont pas présents dans le cache Colab.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Identifiznt du modèle sur Hugging Face
MODEL_ID = "Qwen/Qwen3-1.7B"

# Chargement du tokenizer
print("=== Chargement du tokenizer ===")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("[OK] Tokenizer chargé.")

# Chargement du modèle
print("\n=== Chargement du modèle ===")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto"
)

# Désactivation les comportements spécifiques à l'entraînement
model.eval()
print("[OK] Modèle chargé sur GPU.")

# Calcul du nombre total de paramètres
total_params = sum(p.numel() for p in model.parameters())

print("\n=== Infos sur le modèle ===")
print("Modèle :", MODEL_ID)
print(f"Paramètres : {total_params}")
print(f"Nombre total de paramètres : {total_params}")
print("Périphérique :", next(model.parameters()).device)

# Mesure de la mémoire GPU occupée
if torch.cuda.is_available():
    allocated_memory = torch.cuda.memory_allocated() / 1024**3
    reserved_memory = torch.cuda.memory_reserved() / 1024**3
    print("\n=== Utilisation de la mémoire GPU ===")
    print(f"Mémoire allouée : {allocated_memory:.2f} Gio")
    print(f"Mémoire réservée : {reserved_memory:.2f} Gio")

print("\n[OK] Modèle prêt pour l'évaluation.")

## **3. Génération des réponses du modèle initial**

**Objectif :** interroger Qwen3-1.7B sur les questions OpenCV sans utiliser de RAG ni de fine-tuning.

**Protocole :**
1. Sélectionner une question du jeu de développement.
2. Construire un prompt avec le format conversationnel de Qwen3.
3. Générer une réponse avec des paramètres contrôlés.
4. Extraire uniquement la réponse produite par le modèle.
5. Vérifier la cohérence du résultat.

**Contraintes expérimentales :**
- Utiliser uniquement les questions `dev` pendant la mise au point.
- Ne pas transmettre la réponse de référence au modèle.
- Conserver le même prompt pour toutes les questions.
- Ne pas injecter de contexte documentaire externe.

**Référence :** section de chargement et de première inférence du notebook 00.

### *3.1. Première génération sur une question de développement*

**Objectif :** vérifier le bon fonctionnement de l'inférence avant de traiter automatiquement le dataset.

**Configuration retenue :**
- Mode réflexion désactivé.
- Génération déterministe (`do_sample=False`).
- Limite de 512 nouveaux tokens.
- Prompt système en français.

**Résultats attendus :**
- Une question du jeu `dev` affichée.
- Une réponse technique générée en français.
- Une réponse distincte du prompt initial.
- Aucun contenu du jeu `test` utilisé.

**Point de vigilance :** une réponse grammaticalement correcte n'est pas nécessairement scientifiquement exacte. Sa qualité sera évaluée séparément.

In [ ]:
import torch

# 1. Sélection d'une question de développement
question_test = sorted(dev_questions, key=lambda question: question["id"])[0]

print("=== Question sélectionnée ===")
print("Identifiant :", question_test["id"])
print("Domaine :",question_test["topic"])
print("Question :", question_test["question"])

# 2. Définition d'un prompt système commun au benchmark
PROMPT_SYSTEM = (
    "Tu es un assisstant technique spécialisé en traitement d'images, OpenCV et vision industrielle. "
    "Réponds en français de manière claire, précise et concise. "
    "Explique les notions nécessaires sans inventer d'informations. Si tu ne sais pas, indique-le."
)

messages = [
    {
        "role": "system",
        "content": PROMPT_SYSTEM,
    },
    {
        "role": "user",
        "content": question_test["question"],
    },
]

# 3. Application du format conversationnel de Qwen3
formatted_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

# 4. Conversion du texte en tokens et transfert sur GPU
inputs = tokenizer([formatted_text], return_tensors="pt").to(model.device)

# 5. Génération de la réponse
MAX_NEW_TOKENS = 512

with torch.inference_mode():
    outputs = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )

# 6. Retrait des tokens du prompt initial
prompt_length = inputs["input_ids"].shape[1]
output_tokens = outputs[0][prompt_length:]

response = tokenizer.decode(output_tokens, skip_special_tokens=True).strip()

# 7. Affichage des résultats
print("\n=== Réponse générée ===")
print(response)

print("\n=== Informations de génération ===")
print("Tokens du prompt :", prompt_length)
print("Tokens générés :", len(output_tokens))

if len(output_tokens) >= MAX_NEW_TOKENS:
    print("[WARNING] Limite de génération atteinte. Vérifier si la réponse est complète.")
else:
    print("[OK] Génération terminée.")

### *3.2. Génération automatique des réponses du jeu de développement*

**Objectif :** automatiser l'inférence sur les 10 questions `dev` du dataset OpenCV.

**Méthode :**
1. Trier les questions par identifiant.
2. Réutiliser le prompt système et les paramètres de génération de la section 3.1.
3. Générer une réponse indépendante pour chaque question.
4. Enregistrer les réponses et le nombre de tokens dans une liste Python.
5. Afficher un résumé de chaque génération.

**Résultats attendus :**
- 10 questions traitées.
- 10 réponses enregistrées en mémoire.
- Identifiants CV001 à CV010.
- Aucun accès au jeu `test` pendant la génération.

**Point de vigilance :** le contenu des réponses reste à évaluer. Une génération réussie ne garantit pas l'exactitude scientifique.

In [ ]:
import torch

# 1. Tri des questions pour fixer l'ordre d'évaluatioon
sorted_questions_dev = sorted(dev_questions, key=lambda question: question["id"])

assert len(sorted_questions_dev) == 10, f"Nombre de questions incorrect"

# 2. Liste pour conserver les résultats
resultats_dev = []
print("\n=== Génération des réponses du dataset ===\n")

# 3. Parcours des questions une par une
for numero, question in enumerate(sorted_questions_dev, start=1):

    # Construction les questions une par une
    messages = [
        {
            "role": "system",
            "content": PROMPT_SYSTEM
        },
        {
            "role": "user",
            "content": question["question"]
        },
    ]

    # Application du format conversationnel Qwen3
    formatted_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    # Conversion du prompt en tenseurs sur GPU
    inputs = tokenizer([formatted_text], return_tensors="pt").to(model.device)

    # Génération de la réponse
    MAX_NEW_TOKENS = 512

    with torch.inference_mode():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )

    # Retrait des tokens du prompt initial
    prompt_length = inputs["input_ids"].shape[1]
    output_tokens = outputs[0][prompt_length:]

    # Reconstruction de la réponse textuelle
    response = tokenizer.decode(output_tokens, skip_special_tokens=True).strip()

    # Enregistrement des résultats
    resultats_dev.append({
        "id": question["id"],
        "topic": question["topic"],
        "question": question["question"],
        "response": response,
        "prompt_tokens": prompt_length,
        "generated_tokens": len(output_tokens),
        "limit_reached" : len(output_tokens) >= MAX_NEW_TOKENS
    })

    # Synthèse
    print(f"[{numero}/10] {question["id"]}")
    print(f"    Domaine : {question["topic"]}")
    print(f"    Tokens générés : {len(output_tokens)}")
    print(f"    Réponse : {response[:250]}")

    if len(response) >= 250:
        print(" ...")

    if len(output_tokens) >= MAX_NEW_TOKENS:
        print(" [WARNING] Limite de génération atteinte. Vérifier si la réponse est complète.")
    print()

# 4. Vérif que toutes les questions ont été traitées
assert len(resultats_dev) == 10, f"Nombre de réponses incorrectes : {len(resultats_dev)}"

print("=== Bilan ===")
print("Questions traitées :", len(resultats_dev))
print("Réponses non vides :", sum(bool(r["response"]) for r in resultats_dev))
print("Générations ayant atteint la limite :", sum(r["limit_reached"] for r in resultats_dev))
print("\n[OK] Générations terminées.")

## **4. Mesures et sauvegarde des résultats**

**Objectif :** conserver les réponses générées et mesurer les performances d'inférence du modèle initial Qwen3-1.7B.

**Méthode :**
1. Sauvegarder les réponses du jeu de développement.
2. Mesurer la latence de génération.
3. Calculer le débit de génération en tokens/s.
4. Mesurer la consommation de mémoire GPU.
5. Conserver les résultats dans des fichiers reproductibles.

**Résultats attendus :**
- Réponses de développement sauvegardées sur Google Drive.
- Mesures de performance quantitatives.
- Fichiers JSON exploitables pour les futures comparaisons.

**Référence :** section 8 du notebook 00 — premières mesures d'inférence.

**Point de vigilance :** les résultats expérimentaux doivent être conservés indépendamment de la session Colab.

### *4.1. Sauvegarde persistante des réponses de développement*

**Objectif :** enregistrer les réponses produites à la section 3.2 pour éviter leur perte lors d'une déconnexion Colab.

**Méthode :**
1. Vérifier que les dix réponses ont été générées.
2. Monter Google Drive.
3. Créer le dossier de résultats du benchmark initial.
4. Enregistrer les réponses et les paramètres expérimentaux au format JSON.
5. Vérifier que le fichier a été correctement écrit.

**Résultats attendus :**
- 10 réponses complètes enregistrées.
- Paramètres de génération conservés.
- Fichier accessible depuis Google Drive.

**Point de vigilance :** seules les réponses générées sont sauvegardées. Les réponses de référence du dataset ne sont pas injectées dans le modèle.


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

# 1. Vérification des réponses
assert "resultats_dev" in globals(), "Executer d'abord le 3.2"
assert len(resultats_dev) == 10, f"Nombre de réponses incorrect"
assert all(resultat["response"].strip() for resultat in resultats_dev), "Toutes les réponses doivent être non-vides"

print("[OK] Dix réponses de développement disponibles")


# 2. Montage Google Drive
drive.mount("/content/drive")

# 3. Définition du dossier de sauvegarde persistant
RESULTS_FOLDER = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = RESULTS_FOLDER / "baseline_dev_responses.json"

# 4. Récup de l'empreinte du dataset figé
manifest_path = PROJECT_DIR / "data/evaluation/freeze_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

# 5. Préparation des métadonnées expérimentales
donnees_sauvegarde = {
    "modele": MODEL_ID,
    "configuration":{
        "precision": "fp16",
        "mode_reflexion": False,
        "echantillonnage": False,
        "max_new_tokens": MAX_NEW_TOKENS,
    },
    "dataset_sha256": manifest["dataset_content_sha256"],
    "jeu_evalue": "dev",
    "date_utc": datetime.now(timezone.utc).isoformat(),
    "nombre_questions": len(resultats_dev),
    "resultats": resultats_dev,
}

# 6. Ecriture des résultats au format JSON
with RESULTS_FILE.open("w", encoding="utf-8") as fichier:
    json.dump(donnees_sauvegarde, fichier, indent=4, ensure_ascii=False)


# 7. Vérification du fichier en le rechargeant
donnees_verifiees = json.loads(RESULTS_FILE.read_text(encoding="utf-8"))

assert donnees_verifiees["nombre_questions"] == 10
assert len(donnees_verifiees["resultats"]) == 10

print("\n=== Sauvegarde du benchmark initial ===")
print(f"Modèle : {donnees_verifiees['modele']}")
print("Reponses sauvegardées :", donnees_verifiees["nombre_questions"])
print("Fichier :", RESULTS_FILE)
print("\n[OK] Enregistrement terminé.")

### *4.2. Mesure de la latence, du débit et de la mémoire GPU*

**Objectif :** mesurer quantitativement les performances d'inférence de Qwen3-1.7B sur le GPU NVIDIA T4.

**Méthode :**
1. Sélectionner une question de développement.
2. Préparer le prompt avec la configuration validée en section 3.
3. Synchroniser CUDA avant de démarrer le chronomètre.
4. Mesurer la durée de génération.
5. Calculer le nombre de tokens générés par seconde.
6. Relever le pic de mémoire GPU allouée par PyTorch.

**Configuration :**
- Modèle : Qwen3-1.7B en FP16.
- Mode réflexion désactivé.
- Génération déterministe.
- Maximum de 512 nouveaux tokens.
- Aucune récupération documentaire externe.

**Résultats attendus :**
- Une latence positive, exprimée en secondes.
- Un débit de génération en tokens/s.
- Un pic de mémoire GPU en Gio.
- Une réponse complète, sans erreur CUDA.

**Référence :** section 8 du notebook 00 — premières mesures de performances.

**Point de vigilance :** les opérations CUDA étant asynchrones, une synchronisation est nécessaire pour mesurer correctement leur durée.

In [ ]:
import time
import torch

# 1. Sélection de la même question que dans la section 3.1
question_mesure = sorted(dev_questions, key=lambda question: question["id"])[0]

# 2. Construction de la conversation
messages = [
    {
        "role": "system",
        "content": PROMPT_SYSTEM,
    },
    {
        "role": "user",
        "content": question_mesure["question"],
    },
]

formatted_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

inputs = tokenizer([formatted_text], return_tensors="pt").to(model.device)

# 3. Prépration de la mesure CUDA
assert torch.cuda.is_available(), "GPU CUDA indisponible."

torch.cuda.synchronize()

# Réinitialisation du compteur du pic mémoire PyTorch
torch.cuda.reset_peak_memory_stats()

# 4. Démarrage du chrono
start = time.perf_counter()

# 5. Génération de la réponse
with torch.inference_mode():
    outputs = model.generate(
        **inputs,  max_new_tokens= MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )

# Attente de la fin ds calculs GPU
torch.cuda.synchronize()

# 6. Arrêt du chrono
end = time.perf_counter()
latency = end - start

# 7. Calcul du débit de génération
prompt_length = inputs["input_ids"].shape[1]
tokens_response = outputs[0][prompt_length:]

tokens_number = len(tokens_response)
generation_throughput = tokens_number / latency

# 8. Relever le pic de mémoire PyTorch
memory_peak = torch.cuda.max_memory_allocated() / 1024**3

# 9. Résultats
print("=== Mesures de perfs ===")
print("Question :", question_mesure["id"])
print(f"Tokens générés : {tokens_number}")
print(f"Latence : {latency:.3f} s")
print(f"Débit : {generation_throughput:.3f} tokens/s")
print(f"Pic mémoire GPU PyTorch : {memory_peak:.2f} Gio")

if tokens_number >= MAX_NEW_TOKENS:
    print("\n[WARNING] Limite de génération atteinte.")
else:
    print("\n[OK] Mesures terminées.")


### *4.3. Mesure des performances sur le jeu de développement*

**Objectif :** étendre les mesures de performance aux dix questions `dev` du dataset OpenCV.

**Méthode :**
1. Trier les questions de développement par identifiant.
2. Réutiliser le prompt système et les paramètres de génération.
3. Mesurer individuellement la latence de chaque réponse.
4. Calculer le débit de génération en tokens/s.
5. Mesurer le pic de mémoire GPU PyTorch.
6. Sauvegarder progressivement les résultats sur Google Drive.

**Métriques :**
- Latence totale de génération (secondes).
- Nombre de tokens générés.
- Débit effectif (tokens/s).
- Pic de mémoire GPU PyTorch (Gio).

**Résultats attendus :**
- Dix réponses accompagnées de leurs métriques.
- Un fichier JSON persistant.
- Aucune question du jeu `test` utilisée.

**Références :**
- Section 3.2 du notebook 01 : génération automatique.
- Section 4.2 du notebook 01 : première mesure GPU.

**Point de vigilance :** cette expérience constitue une mesure préliminaire sur le jeu de développement. Les performances finales seront évaluées selon un protocole reproductible sur le jeu de test.

In [ ]:
import json
import time
import torch

from datetime import datetime, timezone
from pathlib import Path

# 1. Réutilisation des données des sections précédentes
dev_questions = sorted(dev_questions, key=lambda item: item["id"])

sys_prompt = PROMPT_SYSTEM
max_new_tokens = MAX_NEW_TOKENS

results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_path = results_dir / "baseline_dev_performance.json"

assert len(dev_questions) == 10
assert torch.cuda.is_available(), "GPU CUDA indisponible."

model.eval()

# 2. Préparation de la liste des mesures
perf_results = []
print("=== Benchmark des 10 questions de développement ===\n")

# 3. Evaluation de chaque question independamment
for index, item in enumerate(dev_questions, start=1):
    messages = [
        {
            "role": "system",
            "content": sys_prompt
        },
        {
            "role": "user",
            "content": item["question"]
        },
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)
    prompt_tokens = model_inputs["input_ids"].shape[1]

    # 4. Mesure de la latence & calcul des métriques
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **model_inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()

    latency = time.perf_counter() - start

    response_tokens = outputs[0][prompt_tokens:]
    generated_tokens = len(response_tokens)

    generation_throughput = generated_tokens / latency
    memory_peak = torch.cuda.max_memory_allocated() / 1024**3

    response_text = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()

    perf_results.append({
        "id": item["id"],
        "topic": item["topic"],
        "question": item["question"],
        "response": response_text,
        "prompt_tokens": prompt_tokens,
        "generated_tokens": generated_tokens,
        "latency": latency,
        "throughput": generation_throughput,
        "peak_gpu_memory": memory_peak,
        "generation_limit_reached": generated_tokens >= max_new_tokens
    })

    # Sauvegarde après chaque question
    with metrics_path.open("w", encoding="utf-8") as file:
        json.dump(
            {
                "model": "Qwen/Qwen3-1.7B",
                "split": "dev",
                "max_new_tokens": max_new_tokens,
                "do_sample": False,
                "enable_thinking": False,
                "updated_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "results": perf_results
            },
            file,
            ensure_ascii=False,
            indent=4
        )

    # Affichage des métriques de la question
    print(f"[{index}/10] {item['id']}")
    print(f"  Latence : {latency:.3f} s")
    print(f"  Débit : {generation_throughput:.2f} tokens/s")
    print(f"  Mémoire GPU : {memory_peak:.2f} Gio")
    print(f"  Tokens générés : {generated_tokens}")

    if generated_tokens >= max_new_tokens:
        print("  [ATTENTION] Limite de génération atteinte.")

    print()

# 4. Vérification du nombre d'évaluations
assert len(perf_results) == 10

print("=== Bilan ===")
print("Questions traitées :", len(perf_results))
print("Réponses non vides :", sum(bool(r["response"]) for r in perf_results))
print("Générations ayant atteint la limite :", sum(r["generation_limit_reached"] for r in perf_results))
print("\n[OK] Mesures terminées & sauvegardées.")

### *4.4. Synthèse statistique et contrôle des générations*

**Objectif :** analyser les performances du modèle sur les dix questions de développement à partir des mesures enregistrées.

**Méthode :**
1. Recharger le fichier JSON des performances.
2. Vérifier que les dix questions sont présentes.
3. Calculer les moyennes, médianes et valeurs extrêmes.
4. Calculer le débit global de génération.
5. Identifier les générations ayant atteint la limite de tokens.
6. Examiner les dernières lignes des réponses concernées.
7. Sauvegarder une synthèse statistique au format JSON.

**Résultats attendus :**
- Statistiques calculées sur les dix questions.
- Questions potentiellement tronquées identifiées.
- Synthèse enregistrée sur Google Drive.

**Référence :** section 4.3 du notebook 01 — mesure des performances.

**Point de vigilance :** une réponse atteignant la limite de génération n'est pas nécessairement incorrecte, mais elle peut être incomplète. Son contenu doit être examiné avant toute notation.

In [ ]:
import json
import statistics

from datetime import datetime, timezone
from pathlib import Path

# 1. Fichier des mesures enregistrées
results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
metrics_path = results_dir / "baseline_dev_performance.json"

assert metrics_path.exists(), "fichier de performances introuvable. Executer d'abord la section 4.2"

# 2. Chargement des mesures
with metrics_path.open(encoding="utf-8") as file:
    benchmark_data = json.load(file)

records = benchmark_data["results"]

assert len(records) == 10, "Nombre de questions incorrectes"
assert len({record["id"] for record in records}) == 10, "Identifiants de questions dupliqués"

print("=== Vérif des données ===")
print("Modèle :", benchmark_data["model"])
print("Questions évaluées :", len(records))


# 3. Extraction des mesures
latencies = [record["latency"] for record in records]
throughputs = [record["throughput"] for record in records]
memory_peaks = [record["peak_gpu_memory"] for record in records]
token_counts = [record["generated_tokens"] for record in records]

# 4. Calcul des statistiques
summary = {
    "question_count": len(records),

    "mean_latency": statistics.mean(latencies),
    "median_latency": statistics.median(latencies),
    "min_latency": min(latencies),
    "max_latency": max(latencies),

    "mean_throughput": statistics.mean(throughputs),
    "median_throughput": statistics.median(throughputs),

    "total_generated_tokens": sum(token_counts),

    "global_throughput": sum(token_counts) / sum(latencies),

    "max_memory_peak": max(memory_peaks),
}

# 5. Affichage des résultats
print("\n=== Statistiques de performances ===")

print(f"Latence moyenne : {summary['mean_latency']:.3f} s")
print(f"Latence médiane : {summary['median_latency']:.3f} s")
print(f"Latence minimale : {summary['min_latency']:.3f} s")
print(f"Latence maximale : {summary['max_latency']:.3f} s")

print(f"Débit moyen : {summary['mean_throughput']:.2f} tokens/s")
print(f"Débit médian : {summary['median_throughput']:.2f} tokens/s")
print(f"Débit global : {summary['global_throughput']:.2f} tokens/s")

print(f"Pic mémoire GPU maximal : {summary['max_memory_peak']:.2f} Gio")

print(f"Nombre de tokens générés : {summary['total_generated_tokens']}")


# 6. Générations limitées
limited_records = [
    record for record in records if record["generation_limit_reached"]
]

summary["limited_generation_ids"] = [record["id"] for record in limited_records]

print("\n=== Contrôle des réponses ===")
print(f"Générations ayant atteint la limite : {len(limited_records)}")

# Affichage de la fin des réponses concernées
for record in limited_records:
    print(f"\nQuestion : {record['id']}")
    print("Derniers caractères de la réponse :")
    print(record["response"][-500:])

# 7. Sauvegarde de la synthèse
summary_path = results_dir / "baseline_dev_summary.json"

summary_data = {
    "model": benchmark_data["model"],
    "split": "dev",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": metrics_path.name,
    "statistics": summary
}

with summary_path.open("w", encoding="utf-8") as file:
    json.dump(summary_data, file, ensure_ascii=False, indent=4)

print("\n=== Sauvegarde ===")
print("Fichier :", summary_path)
print("\n[OK] Synthèse statistique sauvegardée.")

### *4.5. Analyse d'une génération interrompue*

**Objectif :** examiner une réponse ayant atteint la limite de génération afin de déterminer si une modification du protocole est nécessaire.

**Méthode :**
1. Recharger les résultats déjà sauvegardés.
2. Extraire la réponse complète de CV010.
3. Comparer son contenu avec la réponse de référence.
4. Identifier les erreurs, les développements inutiles et les passages incomplets.
5. Déterminer si le prompt doit être ajusté.

**Résultats attendus :**
- Réponse complète de CV010 affichée.
- Points clés de référence disponibles pour comparaison.
- Diagnostic de la troncature.

**Point de vigilance :** une éventuelle modification du prompt devra être testée sur le jeu `dev` et appliquée de manière identique à toutes les questions. Les résultats précédents seront conservés.


In [ ]:
import json
from pathlib import Path

# 1. Rechargement des resultats
metrics_path = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline/baseline_dev_performance.json")

with metrics_path.open("r", encoding="utf-8") as file:
    benchmark_data = json.load(file)

# 2. Génération CV010
result = next(record for record in benchmark_data["results"] if record["id"] == "CV010")
reference = next(record for record in dev_questions if record["id"] == "CV010")

# 3. Affichage de la question et de la réponse générée
print("=== Question CV 010 ===")
print(reference["question"])

print("\n=== Réponse complète du modèle ===")
print(result["response"])

print("\n=== Réponse de référence ===")
print(reference["reference_answer"])

print("\n=== Points clés attendus ===")
for point in reference["key_points"]:
    print(f"- {point}")

# 5. Vérif de la limite de génération
print("\n=== Diagnostic ===")
print("Tokens générés :", result["generated_tokens"])
print("Limite de génération atteinte :", result["generation_limit_reached"])

print("\n[OK] Inspection terminée.")

## **5. Évaluation de la qualité des réponses**

**Objectif :** identifier les erreurs factuelles et les réponses incomplètes du modèle initial, puis étudier l'effet d'une configuration de prompt plus précise.

**Méthode :**
1. Comparer les réponses générées avec les références OpenCV.
2. Identifier les erreurs factuelles et les hallucinations.
3. Tester une variante du prompt uniquement sur le jeu `dev`.
4. Comparer les résultats sans écraser les expériences précédentes.
5. Fixer la configuration avant l'évaluation du jeu `test`.

**Références :**
- Section 4.5 du notebook 01 : diagnostic de CV010.
- Protocole d'évaluation documenté dans `docs/evaluation_protocol.fr.md`.

**Point de vigilance :** le prompt doit rester général et ne doit pas contenir les réponses de référence.

### *5.1. Test d'un prompt plus concis sur CV010*

**Objectif :** vérifier si une consigne de réponse plus courte permet de limiter les développements inutiles et les générations tronquées.

**Configuration :**
- Modèle Qwen3-1.7B inchangé.
- Précision FP16.
- Mode réflexion désactivé.
- Génération déterministe.
- Limite conservée à 512 nouveaux tokens.
- Nouveau prompt système applicable à toutes les questions.

**Méthode :**
1. Réutiliser CV010, appartenant au jeu de développement.
2. Construire un prompt système plus concis.
3. Exécuter une nouvelle génération.
4. Comparer sa qualité à la réponse initiale.
5. Sauvegarder séparément le résultat.

**Résultats attendus :**
- Réponse plus courte et ciblée.
- Pas de code inutile.
- Aucune troncature, idéalement.
- Possibilité de comparer les deux versions du prompt.

**Point de vigilance :** un prompt plus court ne garantit pas une réponse scientifiquement correcte.

In [ ]:
import json
import torch
import time

from datetime import datetime, timezone
from pathlib import Path

# 1. Définition d'une nouvelle consigne générale
system_prompt_v2 = (
    "Tu es un assistant technique spécialisé en traitement d'images, OpenCV & vision industrielle. "
    "Réponds en français, de façon précise et concise, en 2 à 4 phrases et 80 mots maximum. "
    "Réponds directement à la question posée. "
    "Si une fonction OpenCV est demandée, cite son nom exact et explique brièvement son fonctionnement. "
    "N'ajoute pas d'exemple de code ni d'étapes détaillées suaf si la question le demande explicitement. "
    "Ne propose pas d'informations incertaines comme des faits. "
    "Si tu ne sais pas, indique-le clairement. "
)

# 2. Sélection de la question CV010 dans le jeu de développement
selected_question = next(item for item in dev_questions if item["id"] == "CV010")
messages_v2 = [
    {
        "role": "system",
        "content": system_prompt_v2
    },
    {
        "role": "user",
        "content": selected_question["question"]
    },
]

# 3. Format conversationnel Qwen3
formatted_prompt_v2 = tokenizer.apply_chat_template(
    messages_v2, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

model_inputs_v2 = tokenizer([formatted_prompt_v2], return_tensors="pt").to(model.device)

# 4. Génération sans changer la limite de tokens
torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    generated_output_v2 = model.generate(
        **model_inputs_v2, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()
latency_v2 = time.perf_counter() - start

# 5. Extraction des tokens générés uniquement
prompt_length_v2 = model_inputs_v2["input_ids"].shape[1]
response_tokens_v2 = generated_output_v2[0][prompt_length_v2:]

response_text_v2 = tokenizer.decode(response_tokens_v2, skip_special_tokens=True).strip()

# 6. Affichage de la nouvelle reponse
print("=== Test du prompt V2 sur CV010 ===")
print("Question :", selected_question["question"])

print("\n=== Réponse générée ===")
print(response_text_v2)

print("\n=== Informations ===")
print("Tokens générés :", len(response_tokens_v2))
print(f"Latence : {latency_v2:.3f} s")
print("Limite atteinte :", len(response_tokens_v2) >= max_new_tokens)

# 7. Sauvegarde sans écraser les résultats précédents
output_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "cv010_prompt_v2.json"

experiment = {
    "model": MODEL_ID,
    "question_id": selected_question["id"],
    "prompt_version": "v2",
    "system_prompt": system_prompt_v2,
    "max_new_tokens": max_new_tokens,
    "do_sample": False,
    "enable_thinking": False,
    "response": response_text_v2,
    "generated_tokens": len(response_tokens_v2),
    "latency": latency_v2,
    "generation_limit_reached": (len(response_tokens_v2) >= 512),
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

with output_path.open("w", encoding="utf-8") as file:
    json.dump(experiment, file, ensure_ascii=False, indent=4)

print("\n[OK] Nouvelle génération sauvegardée.")

### *5.2. Comparaison des configurations de prompt sur le jeu de développement*

**Objectif :** comparer le comportement de Qwen3-1.7B avec deux prompts système différents sur les mêmes dix questions.

**Configurations :**
- **V1 :** prompt initial, réponses relativement libres.
- **V2 :** prompt plus concis, réponses limitées à quelques phrases.
- Modèle et paramètres de génération inchangés.
- Aucun RAG ni fine-tuning.

**Méthode :**
1. Charger les dix questions `dev`.
2. Générer les réponses avec le prompt V2.
3. Mesurer la latence, le débit et la mémoire GPU.
4. Sauvegarder les résultats dans un fichier distinct.
5. Comparer ensuite les deux versions selon leur qualité et leur efficacité.

**Résultats attendus :**
- Dix réponses V2 sauvegardées.
- Métriques disponibles pour chaque question.
- Résultats V1 conservés.
- Possibilité de reprendre l'exécution après une interruption.

**Point de vigilance :** une réduction du nombre de tokens ne suffit pas à démontrer une amélioration de qualité. Les réponses seront évaluées séparément, selon le protocole établi.

In [ ]:
import json
import time
import torch

from datetime import datetime, timezone
from pathlib import Path

# 1. Préparation des données et du chemin de sauvegarde
dev_questions = sorted(dev_questions, key=lambda item: item["id"])

results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "baseline_dev_prompt_v2_corrected.json"

assert len(dev_questions) == 10, "Nombre de questions incorrectes"
assert torch.cuda.is_available(), "Pas de GPU disponible"
assert system_prompt_v2.strip()

# 2. Empreinte du dataset
manifest_path = PROJECT_DIR / "data/evaluation/freeze_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

# 3. Fixer les params de l'expérience
configuration = {
    "model": MODEL_ID,
    "split": "dev",
    "prompt_version": "v2",
    "system_prompt": system_prompt_v2,
    "max_new_tokens": max_new_tokens,
    "do_sample": False,
    "enable_thinking": False,
    "dataset_sha256": manifest["dataset_content_sha256"]
}

# 4. Reprise d'une expérience partielle (si elle existe)
if output_path.exists():
    previous_data = json.loads(output_path.read_text(encoding="utf-8"))
    assert previous_data["configuration"] == configuration, "La configuration de l'expérience a changé"

    v2_results = previous_data["results"]
else:
    v2_results = []

completed_ids = {item["id"] for item in v2_results}
assert len(completed_ids) == len(v2_results), "Identifiants de questions dupliqués"

model.eval()

print("\n=== Benchmark du prompt V2 ===\n")

# 5. Génération des réponses manquantes
for index, item in enumerate(dev_questions, start=1):
    if item["id"] in completed_ids:
        print(f"[{index}/10] {item["id"]} :  déjà évaluée")
        continue

    print(f"[{index}/10] {item["id"]} :  en cours")

    messages_v2 = [
        {
            "role": "system",
            "content": system_prompt_v2
        },
        {
            "role": "user",
            "content": item["question"]
        },
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages_v2, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)

    prompt_tokens = model_inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    with torch.inference_mode():
        generated_output = model.generate(
            **model_inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()

    # 6. Métriques
    latency = time.perf_counter() - start
    response_tokens = generated_output[0][prompt_tokens:]
    generated_tokens = len(response_tokens)
    throughput = generated_tokens / latency
    response_text = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
    peak_memory = (torch.cuda.max_memory_allocated() / 1024**3)

    # 7. Conservation de la réponse & des mesures
    v2_results.append({
        "id": item["id"],
        "topic": item["topic"],
        "question": item["question"],
        "response": response_text,
        "prompt_tokens": prompt_tokens,
        "generated_tokens": generated_tokens,
        "latency": latency,
        "throughput": throughput,
        "peak_gpu_memory": peak_memory,
        "generation_limit_reached": (generated_tokens >= 512)
    })

    completed_ids.add(item["id"])

    # 8. Sauvegarde des résultats
    output_data = {
        "configuration": configuration,
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": v2_results
    }

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(output_data, file, ensure_ascii=False, indent=4)

    print(f"[{index}/10] {item["id"]}")
    print(f"    Tokens générés : {generated_tokens}")
    print(f"    Latence : {latency:.3f} s")
    print(f"    Débit : {throughput:.2f} tokens/s")

    if generated_tokens >= 512:
        print("  [WARNING] Limite de génération atteinte.")

    print()

# 9. Vérif de la fin du benchmark
assert len(v2_results) == 10
assert all(record["response"] for record in v2_results)

print("Limites atteintes :", sum(record["generation_limit_reached"] for record in v2_results))
print("Fichier :", output_path)

print("\n[OK] Benchmark V2 enregistré.")

In [ ]:
# Vérifier les réponses de l'expérience V2 corrigée.
print("=== Contrôle de l'expérience V2 corrigée ===")

assert len(v2_results) == 10

for result in sorted(v2_results, key=lambda item: item["id"]):
    print(
        f"{result['id']} : "
        f"{result['generated_tokens']} tokens"
    )

# Examiner spécifiquement CV010.
cv010_result = next(
    item for item in v2_results
    if item["id"] == "CV010"
)

print("\n=== Réponse CV010 corrigée ===")
print(cv010_result["response"])

print("\n[OK] Contrôle terminé.")


### *5.3. Comparaison scientifique des réponses V1 et V2*

**Objectif :** comparer les réponses des deux configurations de prompt avec les références techniques du dataset OpenCV.

**Méthode :**
1. Charger les résultats V1 et V2 depuis Google Drive.
2. Vérifier la correspondance des identifiants.
3. Sélectionner une question de développement.
4. Afficher les deux réponses complètes.
5. Afficher la réponse de référence et les points clés attendus.
6. Préparer la notation manuelle.

**Critères d'évaluation :**
- Exactitude technique : 0 à 3 points.
- Complétude : 0 à 2 points.
- Hallucination ou erreur factuelle : oui/non.
- Score total : 0 à 5 points.

**Résultat attendu :** disposer de toutes les informations nécessaires pour attribuer un score justifié à chaque réponse.

**Point de vigilance :** la rapidité, la longueur et la fluidité d'une réponse ne doivent pas être confondues avec son exactitude scientifique.

In [ ]:
import json
from pathlib import Path

# 1. Chemins des expériences enregistrées
results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
v1_path = results_dir / "baseline_dev_performance.json"
v2_path = results_dir / "baseline_dev_prompt_v2_corrected.json"

# 2. Chargement de deux expériences
with v1_path.open("r", encoding="utf-8") as file:
    v1_data = json.load(file)

with v2_path.open("r", encoding="utf-8") as file:
    v2_data = json.load(file)

# 3. Indexation des réponses par identifiant
v1_results = {item["id"]: item for item in v1_data["results"]}
v2_results = {item["id"]: item for item in v2_data["results"]}
references = {item["id"]: item for item in dev_questions}

# 4. Vérif de la comparabilité des expériences
assert set(v1_results) == set(v2_results)
assert set(v1_results) == set(references)
assert len(v1_results) == 10

print("[OK] 10 questions communes aux 2 expériences.")

# 5. Sélection d'une question à inspecter
selected_id = "CV010"

reference = references[selected_id]
v1_response = v1_results[selected_id]
v2_response = v2_results[selected_id]

# 6. Affichage des données pour l'évaluation
print("\n=== Question ===")
print(reference["question"])

print("\n=== Réponse V1 ===")
print(v1_response["response"])

print("\n=== Réponse V2 ===")
print(v2_response["response"])

print("\n=== Réponse de référence ===")
print(reference["reference_answer"])

print("\n=== Points clés attendus ===")

for point in reference['key_points']:
    print("-", point)

print("\n=== Comparaison des générations ===")

print("Tokens V1 :", v1_response["generated_tokens"], "| Tokens V2 :", v2_response["generated_tokens"])

print("Limite atteinte V1 :", v1_response["generation_limit_reached"])
print("Limite atteinte V2 :", v2_response["generation_limit_reached"])

print("\n[OK] Comparaison prête pour la notation.")

### *5.4. Diagnostic du format conversationnel et de l'extraction des réponses*

**Objectif :** identifier l'origine de la reproduction partielle du prompt dans les réponses V2.

**Méthode :**
1. Préparer directement les tokens avec `apply_chat_template`.
2. Générer une nouvelle réponse sur CV010.
3. Vérifier que la sortie contient bien le prompt initial suivi des nouveaux tokens.
4. Extraire uniquement les tokens générés.
5. Rechercher d'éventuels extraits du prompt dans la réponse.

**Configuration :**
- Qwen3-1.7B en FP16.
- Prompt système V2 inchangé.
- Mode réflexion désactivé.
- Génération déterministe.
- Maximum de 512 nouveaux tokens.

**Résultat attendu :** une réponse contenant uniquement le texte produit par l'assistant, sans reproduction du prompt système ni des rôles conversationnels.

**Point de vigilance :** les anciennes expériences sont conservées. Cette étape est un diagnostic, pas encore une nouvelle évaluation.

In [ ]:
import torch

# 1. Reprendre CV010 et le prompt V2 inchangé.
selected_question = next(item for item in dev_questions if item["id"] == "CV010")

messages = [
    {"role": "system", "content": system_prompt_v2},
    {"role": "user", "content": selected_question["question"]}
]

# 2. Appliquer le template et tokeniser en une seule étape.
model_inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_dict=True, return_tensors="pt").to(model.device)

input_ids = model_inputs["input_ids"]
prompt_length = input_ids.shape[-1]

print("=== Contrôle du prompt ===")
print("Tokens du prompt :", prompt_length)

# 3. Générer une réponse avec la même configuration.
model.eval()

with torch.inference_mode():
    generated_output = model.generate(
        **model_inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )

# 4. Vérifier le préfixe des tokens retournés.
prompt_prefix_matches = torch.equal(generated_output[0, :prompt_length], input_ids[0])

print("\n=== Contrôle de la génération ===")
print("Préfixe du prompt conservé :", prompt_prefix_matches)

assert prompt_prefix_matches, "La sortie ne commence pas par les tokens du prompt."


# 5. Extraire uniquement les nouveaux tokens.
response_ids = generated_output[0, prompt_length:]
response_text = tokenizer.decode(response_ids,skip_special_tokens=True).strip()

# 6. Rechercher les signes d'une reproduction du prompt.
prompt_fragments = [
    "si la question le demande explicitement",
    "Ne propose pas d'informations incertaines",
    "\nuser\n",
    "\nassistant\n"
]

prompt_echo_detected = any(fragment in response_text for fragment in prompt_fragments)

print("Tokens générés :", len(response_ids))
print("Reproduction du prompt détectée :", prompt_echo_detected)

print("\n=== Réponse obtenue ===")
print(response_text)

if prompt_echo_detected:
    print("\n[ATTENTION] La réponse contient encore des éléments du prompt.")
else:
    print("\n[OK] Aucun écho évident du prompt détecté.")

### *5.5. Notation scientifique des réponses V1 et V2*

**Objectif :** comparer la qualité technique des réponses produites par les deux configurations de prompt.

**Méthode :**
1. Charger les résultats V1 et V2 corrigés.
2. Récupérer les réponses de référence du dataset.
3. Comparer les deux réponses pour une même question.
4. Évaluer leur exactitude, leur complétude et leurs erreurs factuelles.
5. Justifier chaque score attribué.
6. Conserver les évaluations pour la synthèse finale.

**Grille de notation :**
- Exactitude technique : 0 à 3 points.
- Complétude : 0 à 2 points.
- Score total : 0 à 5 points.
- Hallucination ou erreur factuelle : oui/non.

**Résultats attendus :**
- Deux configurations comparées pour chaque question.
- Scores justifiés par les réponses de référence.
- Identification des erreurs techniques et des informations manquantes.

**Références :**
- Section 5.4 du notebook 01 : correction de l'extraction.
- `docs/evaluation_protocol.fr.md` : protocole de notation.

**Point de vigilance :** l'évaluation de la qualité ne doit pas être fondée uniquement sur la présence de mots-clés ou sur la longueur des réponses.


In [ ]:
import json
from pathlib import Path

# 1. Définir les chemins des résultats.
results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")

v1_path = results_dir / "baseline_dev_performance.json"
v2_path = (results_dir / "baseline_dev_prompt_v2_corrected.json")

# 2. Charger les réponses des deux expériences.
with v1_path.open("r", encoding="utf-8") as file:
    v1_data = json.load(file)

with v2_path.open("r", encoding="utf-8") as file:
    v2_data = json.load(file)

# 3. Indexer les réponses par identifiant.
v1_results = {item["id"]: item for item in v1_data["results"]}
v2_results = {item["id"]: item for item in v2_data["results"]}
references = {item["id"]: item for item in dev_questions}

# 4. Vérifier la correspondance des dix questions.
assert set(v1_results) == set(v2_results)
assert set(v1_results) == set(references)
assert len(v1_results) == 10

print("[OK] Dix questions disponibles pour la notation.")

# 5. Commencer par deux questions représentatives.
selected_ids = ["CV001", "CV005"]

for question_id in selected_ids:

    reference = references[question_id]

    print("\n" + "=" * 70)
    print("QUESTION :", question_id)
    print("=" * 70)

    print("\n=== Énoncé ===")
    print(reference["question"])

    print("\n=== Réponse V1 ===")
    print(v1_results[question_id]["response"])

    print("\n=== Réponse V2 corrigée ===")
    print(v2_results[question_id]["response"])

    print("\n=== Réponse de référence ===")
    print(reference["reference_answer"])

    print("\n=== Points clés attendus ===")

    for point in reference["key_points"]:
        print("-", point)

print("\n[OK] Comparaison prête pour la notation.")

In [ ]:
import json
from pathlib import Path

# 1. Définir les scores provisoires.
manual_scores = [
    {
        "id": "CV001",
        "version": "v1",
        "accuracy": 3,
        "completeness": 2,
        "factual_error": False,
        "justification": "Ouverture morphologique correctement expliquée."
    },
    {
        "id": "CV001",
        "version": "v2",
        "accuracy": 0,
        "completeness": 0,
        "factual_error": True,
        "justification": "Opération incorrecte : erosion inverse."
    },
    {
        "id": "CV005",
        "version": "v1",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Principe partiellement correct, mais pondération "
            "spatiale absente et explication imprécise."
        )
    },
    {
        "id": "CV005",
        "version": "v2",
        "accuracy": 0,
        "completeness": 0,
        "factual_error": True,
        "justification": "Mécanisme de pondération incorrect."
    },
    {
        "id": "CV010",
        "version": "v1",
        "accuracy": 0,
        "completeness": 0,
        "factual_error": True,
        "justification": "Confusion entre minAreaRect et boundingRect."
    },
    {
        "id": "CV010",
        "version": "v2",
        "accuracy": 0,
        "completeness": 0,
        "factual_error": True,
        "justification": "Confusion entre minAreaRect et boundingRect."
    }
]

# 2. Vérifier les scores et calculer les totaux.
for item in manual_scores:
    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert item["id"] in references
    assert item["version"] in ("v1", "v2")

    item["total_score"] = item["accuracy"] + item["completeness"]

assert len({(item["id"], item["version"]) for item in manual_scores}) == len(manual_scores)

# 3. Sauvegarder les annotations provisoires.
scores_path = results_dir / "baseline_dev_quality_scores_draft.json"

with scores_path.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "split": "dev", "status": "provisional", "max_score": 5, "evaluations": manual_scores
        },
        file, ensure_ascii=False, indent=2
    )

print("[OK] Premières notations sauvegardées.")
print("Réponses notées :", len(manual_scores))
print("Fichier :", scores_path)


In [ ]:
selected_ids = ["CV002", "CV003"]

for question_id in selected_ids:
    reference = references[question_id]

    print("\n" + "=" * 65)
    print("QUESTION :", question_id)
    print("=" * 65)

    print("\n=== Énoncé ===")
    print(reference["question"])

    print("\n=== Réponse V1 ===")
    print(v1_results[question_id]["response"])

    print("\n=== Réponse V2 corrigée ===")
    print(v2_results[question_id]["response"])

    print("\n=== Réponse de référence ===")
    print(reference["reference_answer"])

    print("\n=== Points clés attendus ===")
    for point in reference["key_points"]:
        print("-", point)

print("\n[OK] Deux nouvelles questions prêtes pour la notation.")

In [ ]:
import json
from pathlib import Path

# 1. Recharger les notations existantes.
scores_path = (results_dir / "baseline_dev_quality_scores_draft.json")

with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

manual_scores = scores_data["evaluations"]

# 2. Préparer les nouvelles évaluations.
new_scores = [
    {
        "id": "CV002",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": "Fermeture identifiée, mais ordre des opérations inversé : érosion avant dilatation."
    },
    {
        "id": "CV002",
        "version": "v2",
        "accuracy": 3,
        "completeness": 2,
        "factual_error": False,
        "justification": "Fermeture correctement identifiée : dilatation suivie d'une érosion."
    },
    {
        "id": "CV003",
        "version": "v1",
        "accuracy": 3,
        "completeness": 2,
        "factual_error": False,
        "justification": "Différence entre seuil global et seuil local correctement expliquée."
    },
    {
        "id": "CV003",
        "version": "v2",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": False,
        "justification": "Explication globalement correcte mais absence de précision sur le voisinage local."
    }
]

# 3. Mettre à jour les notes par identifiant et version.
# Cette méthode évite de créer des doublons en cas de relance.
scores_by_key = {(item["id"], item["version"]): item for item in manual_scores}

for item in new_scores:
    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2

    item["total_score"] = item["accuracy"] + item["completeness"]

    scores_by_key[(item["id"], item["version"])] = item

# 4. Reconstruire les évaluations dans un ordre stable.
manual_scores = sorted(scores_by_key.values(), key=lambda item: (item["id"], item["version"]))
scores_data["evaluations"] = manual_scores

# 5. Sauvegarder les notations actualisées.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(scores_data, file, ensure_ascii=False, indent=2)

# 6. Vérifier les totaux.
for version in ("v1", "v2"):
    version_scores = [item for item in manual_scores if item["version"] == version]
    total_score = sum(item["total_score"] for item in version_scores)

    print(
        f"Prompt {version.upper()} : "
        f"{len(version_scores)} réponses notées, "
        f"{total_score}/{5 * len(version_scores)} points"
    )

assert len(manual_scores) == 10

print("\n[OK] Notations actualisées et sauvegardées.")

In [ ]:
selected_ids = ["CV004", "CV006"]

for question_id in selected_ids:
    reference = references[question_id]

    print("\n" + "=" * 65)
    print("QUESTION :", question_id)
    print("=" * 65)

    print("\n=== Énoncé ===")
    print(reference["question"])

    print("\n=== Réponse V1 ===")
    print(v1_results[question_id]["response"])

    print("\n=== Réponse V2 corrigée ===")
    print(v2_results[question_id]["response"])

    print("\n=== Réponse de référence ===")
    print(reference["reference_answer"])

    print("\n=== Points clés attendus ===")

    for point in reference["key_points"]:
        print("-", point)

print("\n[OK] Deux nouvelles questions prêtes pour la notation.")


In [ ]:
import json
from pathlib import Path

# 1. Recharger les annotations existantes.
scores_path = results_dir / "baseline_dev_quality_scores_draft.json"

with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

# 2. Définir les nouvelles notations.
new_scores = [
    {
        "id": "CV004",
        "version": "v1",
        "accuracy": 3,
        "completeness": 2,
        "factual_error": False,
        "justification": "Taille carrée, calcul médian et contrainte impaire correctement expliqués. Justification mathématique de la taille impaire légèrement imprécise."
    },
    {
        "id": "CV004",
        "version": "v2",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": "Taille du noyau et contrainte impaire reconnues, mais calcul médian absent. Une taille paire provoque une erreur OpenCV et non de simples artefacts."
    },
    {
        "id": "CV006",
        "version": "v1",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": "Différence automatique/manuelle correcte, mais critère d'optimisation d'Otsu incorrect."
    },
    {
        "id": "CV006",
        "version": "v2",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": "Différence principale correcte, mais absence du critère de variance intra-classe et généralisation excessive sur la fiabilité."
    }
]

# 3. Indexer les notations pour éviter les doublons.
scores_by_key = {(item["id"], item["version"]): item for item in scores_data["evaluations"]}

# 4. Vérifier et intégrer les nouvelles évaluations.
for item in new_scores:
    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2

    item["total_score"] = item["accuracy"] + item["completeness"]

    scores_by_key[(item["id"], item["version"])] = item

# 5. Trier les résultats.
scores_data["evaluations"] = sorted(scores_by_key.values(), key=lambda item: (item["id"], item["version"]))

# 6. Sauvegarder le fichier actualisé.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(scores_data, file, ensure_ascii=False, indent=2)

# 7. Afficher le bilan.
for version in ("v1", "v2"):
    version_scores = [item for item in scores_data["evaluations"] if item["version"] == version]
    total_score = sum(item["total_score"] for item in version_scores)

    print(
        f"Prompt {version.upper()} : "
        f"{len(version_scores)} réponses notées, "
        f"{total_score}/{5 * len(version_scores)} points"
    )

assert len(scores_data["evaluations"]) == 14

print("\n[OK] Quatorze évaluations sauvegardées.")

In [ ]:
selected_ids = ["CV007", "CV008", "CV009"]

for question_id in selected_ids:
    reference = references[question_id]

    print("\n" + "=" * 65)
    print("QUESTION :", question_id)
    print("=" * 65)

    print("\n=== Énoncé ===")
    print(reference["question"])

    print("\n=== Réponse V1 ===")
    print(v1_results[question_id]["response"])

    print("\n=== Réponse V2 corrigée ===")
    print(v2_results[question_id]["response"])

    print("\n=== Réponse de référence ===")
    print(reference["reference_answer"])

    print("\n=== Points clés attendus ===")

    for point in reference["key_points"]:
        print("-", point)

print("\n[OK] Dernières questions prêtes pour la notation.")

In [ ]:
import json

# 1. REcharger les 14 annotations existantes
scores_path = results_dir / "baseline_dev_quality_scores_draft.json"

with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

assert len(scores_data["evaluations"]) == 14

# 2. Préparer les 6 dernières évaluations
new_scores = [
    {
        "id": "CV007",
        "version": "v1",
        "accuracy": 1,
        "completeness": 0,
        "factual_error": True,
        "justification": "Rôle des seuils confus et absence du critère de connexion aux contours forts."
    },
    {
        "id": "CV007",
        "version": "v2",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": "Seuils fort/faible identifiés, mais connectivité absente et confusion entre intensité du gradient et luminosité."
    },
    {
        "id": "CV008",
        "version": "v1",
        "accuracy": 2,
        "completeness": 2,
        "factual_error": True,
        "justification": "Séparation objets/fond et extraction des frontières correctes, mais généralisations incorrectes sur la précision et les performances."
    },
    {
        "id": "CV008",
        "version": "v2",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": False,
        "justification": "Principe correct, mais séparation premier plan/arrière-plan insuffisamment explicite."
    },
    {
        "id": "CV009",
        "version": "v1",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": "contourArea correctement identifié, mais erreur sur le calcul automatique de l'aire de plusieurs fragments."
    },
    {
        "id": "CV009",
        "version": "v2",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": False,
        "justification": "Fonction correcte, unité imprécise (pixels au lieu de pixels carrés) et absence du filtrage par taille."
    }
]

# 3. Indexer les annotations existantes
scores_by_key = {(item["id"], item["version"]): item for item in scores_data["evaluations"]}

# 4. Vérifier et ajouter les nouvelles notes.
for item in new_scores:
    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert isinstance(item["factual_error"], bool)

    item["total_score"] = item["accuracy"] + item["completeness"]

    scores_by_key[(item["id"], item["version"])] = item

# 5. Trier les annotations.
scores_data["evaluations"] = sorted(scores_by_key.values(), key=lambda item: (item["id"], item["version"]))

# 6. Vérifier que toutes les réponses sont notées.
assert len(scores_data["evaluations"]) == 20

expected_ids = {item["id"] for item in dev_questions}

for version in ("v1", "v2"):
    evaluated_ids = {item["id"] for item in scores_data["evaluations"] if item["version"] == version}
    assert evaluated_ids == expected_ids

# 7. Sauvegarder le fichier actualisé.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(scores_data, file, ensure_ascii=False, indent=2)

# 8. Afficher les résultats cumulés.
print("=== Bilan des notations du jeu dev ===")

for version in ("v1", "v2"):
    version_scores = [item for item in scores_data["evaluations"] if item["version"] == version]

    total_score = sum(item["total_score"] for item in version_scores)
    error_count = sum(item["factual_error"] for item in version_scores)

    print(f"\nPrompt {version.upper()}")
    print(f"Questions notées : {len(version_scores)}/10")
    print(f"Score total : {total_score}/50")
    print(f"Erreurs factuelles signalées : {error_count}")

print("\nFichier :", scores_path)
print("[OK] Vingt évaluations sauvegardées.")

### *5.6. Validation des scores et sélection de la configuration finale*

**Objectif :** fixer la configuration de référence avant l'évaluation sur le jeu de test.

**Méthode :**
1. Recharger les vingt notations du jeu `dev`.
2. Vérifier leur cohérence et leur couverture.
3. Calculer les scores des prompts V1 et V2.
4. Sélectionner le prompt selon la qualité technique globale.
5. Enregistrer les scores validés et la configuration retenue.

**Configuration sélectionnée :**
- Modèle : Qwen3-1.7B.
- Prompt système : V1.
- Précision : FP16.
- Mode réflexion : désactivé.
- Génération déterministe.
- Maximum de 512 nouveaux tokens.

**Résultats attendus :**
- `baseline_dev_quality_scores_final.json`
- `baseline_selected_config.json`

**Point de vigilance :** les 40 questions du jeu `test` doivent rester inutilisées jusqu'au verrouillage de la configuration.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

# 1. Résultats enregistrés
results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
draft_path = results_dir / "baseline_dev_quality_scores_draft.json"

with draft_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

evaluations = scores_data["evaluations"]

# 2. Vérif de la couverture des 10 questions
expected_ids = {item["id"] for item in dev_questions}
expected_keys = {(question_id, version) for question_id in expected_ids for version in ("v1", "v2")}
actual_keys = {(item["id"], item["version"]) for item in evaluations}

assert len(evaluations) == 20
assert actual_keys == expected_keys

# 3. Vérif de la cohérence des notes
for item in evaluations:
    assert isinstance(item["accuracy"], int)
    assert isinstance(item["completeness"], int)
    assert isinstance(item["factual_error"], bool)

    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2

    assert item["total_score"] == item["accuracy"] + item["completeness"]
    assert item['justification'].strip()

# 4. Résultats par version
version_summary = {}

for version in ("v1", "v2"):
    version_scores = [item for item in evaluations if item["version"] == version]

    version_summary[version] = {
        "question_count": len(version_scores),
        "total_score": sum(item["total_score"] for item in version_scores),
        "factual_error_count": sum(item["factual_error"] for item in version_scores)
    }

print("=== Bilan validé ===")

for version, summary in version_summary.items():
    print(
        f"\nPrompt {version.upper()} : "
        f"{summary["total_score"]} / 50 points, "
        f"{summary["factual_error_count"]} erreurs factuelles"
    )

# 5. Sélection de la configuration ayant le meeilleur score
selected_version = max(version_summary, key=lambda version: version_summary[version]["total_score"])
assert selected_version in ("v1", "v2")

# 6. Empreinte du dataset figé
manifest_path = PROJECT_DIR / "data/evaluation/freeze_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
dataset_hash = manifest["dataset_content_sha256"]

# 7. Sauvegarde des notes validées dans un fivhier distinct
final_scores_path = results_dir / "baseline_dev_quality_scores_final.json"

final_scores_data = {
    **scores_data,
    "status": "final",
    "validated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_sha256": dataset_hash,
    "summary": version_summary
}

with final_scores_path.open("w", encoding="utf-8") as file:
    json.dump(final_scores_data, file, ensure_ascii=False, indent=2)

# 8. Enregistrement de la config sélectionnée
selected_config = {
    "model_id": MODEL_ID,
    "prompt_version": selected_version,
    "system_prompt": PROMPT_SYSTEM,
    "dtype": "float16",
    "enable_thinking": False,
    "do_sample": False,
    "max_new_tokens": MAX_NEW_TOKENS,
    "selected_split": "dev",
    "selected_metric": "total_quality_score",
    "dataset_sha256": dataset_hash,
    "source_results": "baseline_dev_performance.json",
    "quality_scores_file": final_scores_path.name,
    "selected_at_utc": datetime.now(timezone.utc).isoformat()
}

config_path = results_dir / "baseline_selected_config.json"

with config_path.open("w", encoding="utf-8") as file:
    json.dump(selected_config, file, ensure_ascii=False, indent=2)

# 9. Vérifier que les fichiers sont bien enregistrés
assert final_scores_path.exists()
assert config_path.exists()

print("\n=== Configuration sélectionnée ===")
print("Modèle :", selected_config["model_id"])
print("Prompt :", selected_version.upper())
print("Limite :", selected_config["max_new_tokens"], "tokens")
print("Mode réflexion :", selected_config["enable_thinking"])

print("\n=== Sauvegarde ===")
print("Scores :", final_scores_path)
print("Configuration :", config_path)
print("\n[OK] Configuration du benchmark verrouillée.")

## **6. Benchmark final sur le jeu de test**

**Objectif :** mesurer les performances et la qualité technique du modèle initial Qwen3-1.7B sur les 40 questions du jeu de test figé.

**Méthode :**
1. Charger la configuration sélectionnée sur le jeu `dev`.
2. Vérifier l'intégrité des paramètres et la séparation des données.
3. Générer les réponses aux 40 questions `test`.
4. Mesurer la latence, le débit et l'utilisation mémoire GPU.
5. Évaluer les réponses selon le protocole défini.
6. Sauvegarder les résultats et produire une synthèse reproductible.

**Configuration :**
- Modèle : Qwen3-1.7B.
- Prompt système : V1.
- Précision : FP16.
- Mode réflexion désactivé.
- Génération déterministe.
- Limite de 512 nouveaux tokens.

**Résultats attendus :**
- 40 réponses techniques.
- Métriques individuelles et agrégées.
- Évaluation de l'exactitude et de la complétude.
- Identification des erreurs factuelles.
- Résultats persistants sur Google Drive.

**Point de vigilance :** les questions de test ne doivent pas servir à modifier le prompt ou les paramètres du modèle. La configuration restera identique pendant toute l'expérience.

### *6.1. Vérification de la configuration et du jeu de test*

**Objectif :** s'assurer que le benchmark final utilise exactement les paramètres sélectionnés en section 5.6.

**Méthode :**
1. Charger le fichier de configuration verrouillée.
2. Comparer son empreinte avec le manifeste du dataset figé.
3. Vérifier la présence des 40 questions `test`.
4. Vérifier leur séparation des dix questions `dev`.
5. Contrôler le modèle chargé et la disponibilité du GPU.

**Résultats attendus :**
- Configuration V1 reconnue.
- 40 questions disponibles sans doublon.
- Aucune question commune entre `dev` et `test`.
- Modèle FP16 chargé sur le GPU.
- Vérifications terminées sans erreur.

**Point de vigilance :** cette cellule vérifie la correspondance des empreintes enregistrées. La vérification cryptographique du contenu du dataset reste assurée par le script de gel utilisé dans le notebook 00 et lors de la préparation des données.

In [ ]:
import json
import torch

from pathlib import Path
from collections import Counter

# 1. Définition des chemins du prjet et des résultats
project_dir = Path(PROJECT_DIR)
results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
config_path = results_dir / "baseline_selected_config.json"
manifest_path = project_dir / "data/evaluation/freeze_manifest.json"

assert config_path.exists(), "Configuration verrouillée introuvable."
assert manifest_path.exists(), "Maifeste du dataset introuvable."

# 2. Chargement de la configuration
with config_path.open("r", encoding="utf-8") as file:
    selected_config = json.load(file)

with manifest_path.open("r", encoding="utf-8") as file:
    freeze_manifest = json.load(file)

print("=== Configuration du benchmark ===")
print("Modèle :", selected_config["model_id"])
print("Prompt :", selected_config["prompt_version"])
print("Précision :", selected_config["dtype"])
print("Maximum de nouveaux tokens :", selected_config["max_new_tokens"], "tokens")

# 3. Vérification que la config n'a pas changé
assert selected_config["model_id"] == MODEL_ID
assert selected_config["prompt_version"] == "v1"
assert selected_config["system_prompt"] == PROMPT_SYSTEM
assert selected_config["dtype"] == "float16"
assert selected_config["enable_thinking"] is False
assert selected_config["do_sample"] is False
assert selected_config["max_new_tokens"] == MAX_NEW_TOKENS
assert selected_config["selected_split"] == "dev"

# 4. Vérif de la cohérence des empreintes enregistrées
assert selected_config["dataset_sha256"] == freeze_manifest["dataset_content_sha256"], "Empreinte du dataset différente"

print("\n[OK] COnfiguration verouillée cohérente")

# 5. Préparation des questions de test
test_records = sorted(test_questions, key=lambda record: record["id"])
dev_ids = {record["id"] for record in dev_questions}
test_ids = {record["id"] for record in test_records}

assert len(test_ids) == 40
assert len(test_records) == 40

assert all(record["split"] == "test" for record in test_records)
assert dev_ids.isdisjoint(test_ids), "Chevauchement entre jeux dev et test"

# 6. Vérif de la répartition des domaines
topic_counts = Counter(record["topic"] for record in test_records)

assert len(topic_counts) == 5
assert all(count == 8 for count in topic_counts.values())

print("\n=== Jeu de test ===")
print("Questions :", len(test_records))
print("Questions communes avec dev :", len(dev_ids & test_ids))

print("\nRépartition des questions par domaine :")
for topic, count in topic_counts.items():
    print(f"    {topic} : {count} questions")

# 7. Vérif du GPU et du modèle actuellement chargé
assert torch.cuda.is_available(), "GPU CUDA indisponible"
assert model.device.type == "cuda", "Modèle non chargé sur le GPU."

model_dtype = next(model.parameters()).dtype
assert model_dtype == torch.float16, "Le modèle n'utilise pas la précision FP16"

model.eval()

print("\n=== Environnement ===")
print("GPU :", torch.cuda.get_device_name(0))
print("Précision du modèle :", model_dtype)

print("\n[OK] Benchmark final prêt.")

In [ ]:
# Vérifier les fichiers définitifs avant le benchmark test.
final_scores_path = results_dir / "baseline_dev_quality_scores_final.json"

with final_scores_path.open("r", encoding="utf-8") as file:
    final_scores = json.load(file)

assert len(final_scores["evaluations"]) == 20
assert final_scores["summary"]["v1"]["total_score"] == 31
assert final_scores["summary"]["v2"]["total_score"] == 23

assert selected_config["source_results"] == "baseline_dev_performance.json"

print("[OK] Vingt évaluations définitives conservées.")
print("[OK] Métadonnées de la configuration corrigées.")
print("[OK] Benchmark test autorisé.")


### *6.2. Génération des réponses sur les 40 questions de test*

**Objectif :** exécuter le modèle Qwen3-1.7B sur le jeu `test` figé avec la configuration V1 sélectionnée sur le jeu `dev`.

**Méthode :**
1. Charger les paramètres depuis le fichier de configuration verrouillé.
2. Parcourir les 40 questions de test par identifiant.
3. Construire les prompts sans inclure les réponses de référence.
4. Générer chaque réponse avec les paramètres retenus.
5. Mesurer la latence, le débit et le pic mémoire GPU.
6. Sauvegarder les résultats après chaque question.
7. Reprendre automatiquement les questions restantes après une interruption.

**Configuration :**
- Modèle : Qwen3-1.7B en FP16.
- Prompt système : V1 verrouillé.
- Génération déterministe.
- Mode réflexion désactivé.
- Maximum de 512 nouveaux tokens.

**Résultats attendus :**
- 40 réponses enregistrées dans un fichier JSON.
- Métriques individuelles.
- Aucune question de développement mélangée au jeu de test.
- Reprise possible en cas de déconnexion Colab.

**Point de vigilance :** ne pas modifier le prompt, la limite de tokens ou les paramètres de génération à partir des réponses du jeu de test

In [ ]:
import json
import torch
import torch

from datetime import timezone, datetime
from pathlib import Path

# 1. Récup de la config verouillée
config_path = results_dir / "baseline_selected_config.json"

with config_path.open("r", encoding="utf-8") as file:
    locked_config = json.load(file)

test_records = sorted(test_questions, key=lambda record: record["id"])
test_ids = [record["id"] for record in test_records]

assert len(test_ids) == 40
assert len(test_records) == 40

assert locked_config["model_id"] == MODEL_ID
assert locked_config["prompt_version"] == "v1"
assert locked_config["enable_thinking"] is False
assert locked_config["do_sample"] is False

assert locked_config['dataset_sha256'] == freeze_manifest['dataset_content_sha256']

assert torch.cuda.is_available(), "GPU CUDA indisponible"
assert next(model.parameters()).dtype == torch.float16, "Le modèle n'utilise pas la précision FP16"

model.eval()

# 2. Utilisation des paramètres du fichier verrouillé
system_prompt = locked_config["system_prompt"]
max_new_tokens = locked_config["max_new_tokens"]

run_config = {
    "model_id": MODEL_ID,
    "prompt_version": "v1",
    "system_prompt": system_prompt,
    "dtype": "float16",
    "enable_thinking": False,
    "do_sample": False,
    "max_new_tokens": max_new_tokens,
    "dataset_sha256": freeze_manifest["dataset_content_sha256"]
}

# 3. Préparer le fichier de résultats
output_path = results_dir / "baseline_test_v1_results.json"

# Fichier temporaire utilisé pendant les sauvegardes
temp_path = results_dir / "baseline_test_v1_results.json.tmp"

if output_path.exists():
    with output_path.open("r", encoding="utf-8") as file:
        checkpoint = json.load(file)

    # Refus d'une reprise avec des paramètres différents.
    assert checkpoint["configuration"] == run_config
    assert checkpoint["split"] == "test"
    assert checkpoint["expected_ids"] == test_ids

    test_results = checkpoint["results"]

    print(f"[INFO] Reprise : {len(test_results)} questions déjà enregistrées.")
else:
    test_results = []
    print("[INFO] Nouvelle expérience de benchmark.")

# 5. Contrôle des résultats déjà enregistrés
completed_ids = {result["id"] for result in test_results}

assert len(completed_ids) == len(test_results)
assert completed_ids.issubset(set(test_ids))

for result in test_results:
    assert result["response"].strip()
    assert result["generated_tokens"] > 0

print("\n=== Benchmark final du modèle initial ===")
print("Modèle :", locked_config["model_id"])
print("Prompt :", locked_config["prompt_version"].upper())
print("Questions prévues :", len(test_records))
print("Questions déjà terminées :", len(test_results))
print()

# 6. Parcourir les questions du jeu de test
for index, record, in enumerate(test_records, start=1):
    question_id = record["id"]

    # Ignorer les questions déjà sauvegardées
    if question_id in completed_ids:
        print(f"{index}/40 {question_id} : déjà enregistrée.")
        continue

    print(f"{index}/40 {question_id} : en cours ...")

    # 7. Construction du prompt sans réponse de reference
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": record["question"]}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)
    prompt_tokens = model_inputs["input_ids"].shape[1]

    # 8. Préparer les mesures GPU
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    # 9. Génération de la reponse
    with torch.inference_mode():
        generated_output = model.generate(
            **model_inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()

    # 8. Métriques
    latency = time.perf_counter() - start
    assert torch.equal(generated_output[0, :prompt_tokens], model_inputs["input_ids"][0]), "Préfixe du prompt incorrect."

    response_tokens = generated_output[0][prompt_tokens:]
    generated_tokens = len(response_tokens)
    throughput = generated_tokens / latency
    response_text = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
    assert response_text, f"Réponse vide pour {question_id}."

    peak_memory = (torch.cuda.max_memory_allocated() / 1024**3)
    generation_limit_reached = generated_tokens >= max_new_tokens


    # 9. Enregistrement de la réponse et  de ses mesures
    test_results.append({
        "id": question_id,
        "topic": record["topic"],
        "question": record["question"],
        "response": response_text,
        "prompt_tokens": prompt_tokens,
        "generated_tokens": generated_tokens,
        "latency": latency,
        "throughput": throughput,
        "peak_gpu_memory": peak_memory,
        "generation_limit_reached": generation_limit_reached
    })

    completed_ids.add(question_id)

    # 10. Sauvegarde persistante
    checkpoint = {
        "configuration": run_config,
        "split": "test",
        "expected_ids": test_ids,
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": test_results
    }

    # Écrire d'abord dans un fichier temporaire.
    # Puis remplacer le fichier principal.
    with temp_path.open("w", encoding="utf-8") as file:
        json.dump(checkpoint, file, ensure_ascii=False, indent=2)

    temp_path.replace(output_path)

    # 11. Afficher l'avancement.
    print(f"  Tokens générés : {generated_tokens}")
    print(f"  Latence : {latency:.3f} s")
    print(f"  Débit : {throughput:.2f} tokens/s")
    print(f"  Pic mémoire : {peak_memory:.2f} Gio")

    if generation_limit_reached:
        print("  [ATTENTION] Limite de génération atteinte.")

    print(f"  [OK] Sauvegardée ({index}/40)\n")


# 12. Vérifier que le benchmark est complet.
assert len(test_results) == 40

assert {item["id"] for item in test_results} == set(test_ids)

print("\n=== Bilan final ===")
print("Questions traitées :", len(test_results))

print("Réponses non vides :", sum(bool(item["response"]) for item in test_results))

print("Générations ayant atteint la limite :", sum(result["generation_limit_reached"] for result in test_results))

print("Fichier :", output_path)
print("\n[OK] Quarante réponses sauvegardées.")

### *6.3. Analyse statistique des performances sur le jeu de test*

**Objectif :** analyser les résultats de génération des 40 questions du jeu `test` sans relancer le modèle.

**Méthode :**
1. Recharger les réponses et métriques sauvegardées.
2. Vérifier l'intégrité des 40 enregistrements.
3. Calculer les statistiques globales de latence, de débit et de mémoire GPU.
4. Comparer les performances des cinq domaines.
5. Identifier les générations ayant atteint la limite de tokens.
6. Sauvegarder une synthèse statistique reproductible.

**Métriques :**
- Latence moyenne, médiane, minimale et maximale.
- Débit moyen et débit global.
- Nombre total de tokens générés.
- Pic mémoire GPU PyTorch maximal.
- Proportion de générations ayant atteint la limite.

**Résultats attendus :**
- Une synthèse globale du benchmark.
- Des statistiques pour chacun des cinq domaines.
- Une liste des générations potentiellement incomplètes.
- Le fichier `baseline_test_v1_summary.json`.

**Point de vigilance :** les performances de génération ne mesurent pas l'exactitude technique des réponses. Cette dernière fera l'objet de la section 6.4.

In [ ]:
import json
import statistics

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict


# 1. Définir les fichiers nécessaires.
results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")

results_path = results_dir / "baseline_test_v1_results.json"
config_path = results_dir / "baseline_selected_config.json"
summary_path = results_dir / "baseline_test_v1_summary.json"

# 2. Recharger les résultats et la configuration.
with results_path.open("r", encoding="utf-8") as file:
    benchmark_data = json.load(file)

with config_path.open("r", encoding="utf-8") as file:
    selected_config = json.load(file)

records = benchmark_data["results"]

# 3. Vérifier l'intégrité des données.
expected_ids = benchmark_data["expected_ids"]
record_ids = [item["id"] for item in records]

assert benchmark_data["split"] == "test"
assert len(expected_ids) == 40
assert len(set(expected_ids)) == 40
assert len(records) == 40
assert len(set(record_ids)) == 40
assert set(record_ids) == set(expected_ids)

# Vérifier que la configuration est toujours identique.
for key, value in benchmark_data["configuration"].items():
    assert selected_config[key] == value

for item in records:
    assert item["response"].strip()
    assert item["generated_tokens"] > 0
    assert item["latency"] > 0
    assert item["throughput"] > 0
    assert item["peak_gpu_memory"] > 0

    assert item["generation_limit_reached"] == (item["generated_tokens"] >= selected_config["max_new_tokens"])

print("[OK] Quarante résultats valides et cohérents.")


# 4. Extraire les mesures.
latencies = [item["latency"] for item in records]
throughputs = [item["throughput"] for item in records]
token_counts = [item["generated_tokens"] for item in records]
memory_peaks = [item["peak_gpu_memory"] for item in records]

total_tokens = sum(token_counts)
total_latency = sum(latencies)

# 5. Identifier les questions ayant atteint la limite.
limited_ids = sorted(item["id"] for item in records if item["generation_limit_reached"])

# 6. Calculer les statistiques globales.
global_stats = {
    "question_count": len(records),
    "total_generated_tokens": total_tokens,
    "mean_generated_tokens": statistics.mean(token_counts),
    "total_latency_seconds": total_latency,
    "mean_latency_seconds": statistics.mean(latencies),
    "median_latency_seconds": statistics.median(latencies),
    "min_latency_seconds": min(latencies),
    "max_latency_seconds": max(latencies),
    "mean_throughput": statistics.mean(throughputs),
    "median_throughput": statistics.median(throughputs),
    "global_throughput": total_tokens / total_latency,
    "max_gpu_memory_gib": max(memory_peaks),
    "generation_limit_count": len(limited_ids),
    "generation_limit_percentage": 100 * len(limited_ids) / len(records),
    "generation_limit_ids": limited_ids
}


# 7. Regrouper les résultats par domaine.
records_by_topic = defaultdict(list)

for item in records:
    records_by_topic[item["topic"]].append(item)

assert len(records_by_topic) == 5
assert all(len(items) == 8 for items in records_by_topic.values())

topic_stats = {}

for topic, items in sorted(records_by_topic.items()):

    topic_tokens = sum(item["generated_tokens"] for item in items)
    topic_latency = sum(item["latency"] for item in items)
    topic_stats[topic] = {
        "question_count": len(items),
        "mean_latency_seconds": statistics.mean(item["latency"] for item in items),
        "global_throughput": topic_tokens / topic_latency,
        "mean_generated_tokens": statistics.mean(item["generated_tokens"] for item in items),
        "generation_limit_count": sum(item["generation_limit_reached"] for item in items)
    }


# 8. Afficher les statistiques globales.
print("\n=== Performances globales ===")

print("Questions :", global_stats["question_count"])
print("Tokens générés :", global_stats["total_generated_tokens"])
print(f"Latence moyenne : {global_stats['mean_latency_seconds']:.3f} s")
print(f"Latence médiane : {global_stats['median_latency_seconds']:.3f} s")
print(f"Débit global : {global_stats['global_throughput']:.2f} tokens/s")
print(f"Pic mémoire GPU : {global_stats['max_gpu_memory_gib']:.2f} Gio")

print(
    f"Limite de génération atteinte : "
    f"{global_stats['generation_limit_count']}/40 "
    f"({global_stats['generation_limit_percentage']:.1f} %)"
)


# 9. Afficher les statistiques par domaine.
print("\n=== Performances par domaine ===")

for topic, stats in topic_stats.items():
    print(f"\n{topic}")
    print(f"  Latence moyenne : {stats['mean_latency_seconds']:.2f} s")
    print(f"  Débit global : {stats['global_throughput']:.2f} tokens/s")
    print(f"  Limites atteintes : {stats['generation_limit_count']}/8")


# 10. Afficher les générations à inspecter.
print("\n=== Questions à inspecter ===")

for question_id in limited_ids:
    print("-", question_id)


# 11. Sauvegarder la synthèse.
summary_data = {
    "model_id": selected_config["model_id"],
    "split": "test",
    "prompt_version": selected_config["prompt_version"],
    "dataset_sha256": selected_config["dataset_sha256"],
    "source_file": results_path.name,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "global_statistics": global_stats,
    "topic_statistics": topic_stats
}

with summary_path.open("w", encoding="utf-8") as file:
    json.dump(summary_data, file, ensure_ascii=False, indent=2)

print("\n=== Sauvegarde ===")
print("Fichier :", summary_path)
print("[OK] Synthèse statistique enregistrée.")


### *6.4. Évaluation scientifique des réponses du jeu de test*

**Objectif :** mesurer la qualité technique des 40 réponses générées par la baseline Qwen3-1.7B.

**Méthode :**
1. Charger les réponses du jeu `test` déjà sauvegardées.
2. Associer chaque réponse à sa référence OpenCV.
3. Examiner les réponses par groupes de cinq questions.
4. Évaluer leur exactitude et leur complétude.
5. Identifier les affirmations techniquement incorrectes.
6. Justifier chaque score et sauvegarder les annotations.
7. Calculer les résultats globaux et par domaine.

**Grille d'évaluation :**

- **Exactitude technique (0 à 3 points) :**
  - 3 : réponse techniquement correcte.
  - 2 : réponse essentiellement correcte, avec des imprécisions.
  - 1 : quelques éléments corrects, mais des erreurs importantes.
  - 0 : réponse incorrecte ou sans rapport.

- **Complétude (0 à 2 points) :**
  - 2 : tous les éléments essentiels sont couverts.
  - 1 : une partie des éléments essentiels est couverte.
  - 0 : éléments essentiels absents.

- **Erreur factuelle :** indicateur booléen indépendant du score.

**Score maximal :** 5 points par question, soit 200 points pour les 40 questions.

**Résultats attendus :**
- 40 évaluations justifiées.
- Score global de qualité.
- Taux de réponses contenant des erreurs factuelles.
- Comparaison des cinq domaines.
- Analyse spécifique des générations ayant atteint la limite.

**Point de vigilance :** aucune modification de la configuration V1 ne sera effectuée après consultation des réponses du jeu de test.

In [ ]:
# Inspection du premier groupe de questions test

import json
from pathlib import Path

# 1. Chemin des réponses du benchmark
results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
results_path = results_dir / "baseline_test_v1_results.json"

# 2. Chargement des réponses sauvegardées
with results_path.open("r", encoding="utf-8") as file:
    benchmark_data = json.load(file)

# 3. Indexation des réponses par identifiant
model_responses = {item['id'] : item for item in benchmark_data['results']}

# 4. Indexation des références du jeu test
test_references = {item['id'] : item for item in test_questions}

# 5. Vérif de la correspondance des 40 questions
assert len(model_responses) == 40
assert len(test_references) == 40
assert set(model_responses) == set(test_references)

print("[OK] Quarante réponses prêtes pour la notation.")

# 6. Sélection des 5 premières réponses
selected_ids = ["CV011", "CV012", "CV013", "CV014", "CV015"]

# 7. Affichage des infos nécessaires pour la notation
for question_id in selected_ids:

    reference = test_references[question_id]
    result = model_responses[question_id]

    print("\n" + "=" *70)
    print(f"QUESTION {question_id}")
    print(f"DOMAINE : {reference['topic']}")
    print("=" *70)

    print("\n=== Enoncé ===")
    print(reference['question'])

    print("\n=== Réponse générée ===")
    print(result['response'])

    print("\n=== Réponse de référence ===")
    print(reference['reference_answer'])

    print("\n=== Points clés attendus ===")

    for point in reference['key_points']:
        print("-", point)

    print("\n=== Infos de génération ===")
    print("Tokens :", result['generated_tokens'])
    print("Limite atteinte :", result['generation_limit_reached'])

    if result['generation_limit_reached']:
        print("[WARNING] Examiner une eventuelle interruption de la réponse.")

print("\n[OK] Premier groupe prêt pour la notation.")

In [ ]:
# Sauvegarde des notations CV011 à CV015

import json
from pathlib import Path

# 1. Fichiers de sauvegarde
results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
scores_path = results_dir / "baseline_test_v1_quality_scores_draft.json"
config_path = results_dir / "baseline_selected_config.json"

# 2. Chargement de la configuration
with config_path.open("r", encoding="utf-8") as file:
    locked_config = json.load(file)

assert locked_config["prompt_version"] == "v1"

# 3. Annotations précédentes si elles existent.
if scores_path.exists():
    with scores_path.open("r", encoding="utf-8") as file:
        scores_data = json.load(file)

    assert scores_data["split"] == "test"
    assert scores_data["prompt_version"] == "v1"
    assert scores_data["dataset_sha256"] == locked_config["dataset_sha256"]
else:
    scores_data = {
        "split": "test",
        "prompt_version": "v1",
        "dataset_sha256": locked_config["dataset_sha256"],
        "status": "provisional",
        "max_score": 5,
        "evaluations": []
    }

# 4. Préparer les cinq premières notes.
new_scores = [
    {
        "id": "CV011",
        "version": "v1",
        "accuracy": 3,
        "completeness": 2,
        "factual_error": False,
        "justification": "Normalisation par 1/25, somme égale à 1 et moyenne locale correctement expliquées."
    },
    {
        "id": "CV012",
        "version": "v1",
        "accuracy": 3,
        "completeness": 2,
        "factual_error": False,
        "justification": "Atténuation du bruit et des petits défauts, perte de détails et choix de la taille du noyau correctement expliqués."
    },
    {
        "id": "CV013",
        "version": "v1",
        "accuracy": 0,
        "completeness": 0,
        "factual_error": True,
        "justification": (
            "Inversion des comportements de THRESH_BINARY "
            "et THRESH_BINARY_INV. Explication centrale incorrecte."
        )
    },
    {
        "id": "CV014",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Rôle de blockSize partiellement correct, "
            "contrainte impaire absente, définition de C "
            "incorrecte et exemple OpenCV invalide."
        )
    },
    {
        "id": "CV015",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Contours correctement identifiés, mais "
            "absence de la différence dilatation-érosion "
            "et description incorrecte de la méthode."
        )
    }
]

# 5. Construire un index pour éviter les doublons.
scores_by_key = {(item["id"], item["version"]): item for item in scores_data["evaluations"]}

assert len(scores_by_key) == len(scores_data["evaluations"])

# 6. Vérifier et intégrer les cinq annotations.
for item in new_scores:
    assert item["id"] in test_references
    assert item["id"] in model_responses
    assert item["version"] == "v1"
    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert isinstance(item["factual_error"], bool)

    item["total_score"] = item["accuracy"] + item["completeness"]
    scores_by_key[(item["id"], item["version"])] = item

# 7. Reconstruire la liste dans l'ordre des identifiants.
scores_data["evaluations"] = sorted(scores_by_key.values(), key=lambda item: item["id"])

# 8. Sauvegarder les annotations.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(scores_data, file, ensure_ascii=False, indent=2)

# 9. Afficher le bilan provisoire.
evaluations = scores_data["evaluations"]

total_score = sum(item["total_score"] for item in evaluations)

error_count = sum(item["factual_error"] for item in evaluations)

print("=== Bilan des notations test ===")
print(f"Questions notées : {len(evaluations)}/40")
print(f"Score cumulé : {total_score}/{5 * len(evaluations)}")
print(f"Erreurs factuelles signalées : {error_count}")

print("\nFichier :", scores_path)
print("[OK] Notations sauvegardées.")

In [ ]:
# Fonction d'inspection des réponses du benchmark

def display_questions(question_ids):
    """
    Afficher les réponses et références pour un groupe
    de questions du benchmark.
    """

    for question_id in question_ids:
        reference = test_references[question_id]
        result = model_responses[question_id]

        print("\n" + "=" * 70)
        print("QUESTION :", question_id)
        print("DOMAINE :", reference["topic"])
        print("=" * 70)

        print("\n=== Énoncé ===")
        print(reference["question"])

        print("\n=== Réponse générée ===")
        print(result["response"])

        print("\n=== Réponse de référence ===")
        print(reference["reference_answer"])

        print("\n=== Points clés attendus ===")
        for point in reference["key_points"]:
            print("-", point)

        print("\n=== Informations de génération ===")
        print("Tokens :", result["generated_tokens"])
        print(
            "Limite atteinte :",
            result["generation_limit_reached"]
        )

        if result["generation_limit_reached"]:
            print(
                "[ATTENTION] Vérifier si la réponse "
                "est interrompue."
            )

    print("\n[OK] Groupe prêt pour la notation.")


# Examiner le deuxième groupe.
selected_ids = ["CV016", "CV017", "CV018", "CV019", "CV020"]

display_questions(selected_ids)

In [ ]:
# Deuxième groupe : CV016 à CV020
selected_ids = ["CV016", "CV017", "CV018", "CV019", "CV020"]

display_questions(selected_ids)

In [ ]:
# Enregistrement des notations CV016 à CV020
import json
from pathlib import Path

# 1. Retrouver le fichier des annotations.
scores_path = results_dir / "baseline_test_v1_quality_scores_draft.json"

assert scores_path.exists(), "Le fichier des premières notations est introuvable."

# 2. Charger les cinq évaluations précédentes.
with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

assert scores_data["split"] == "test"
assert scores_data["prompt_version"] == "v1"

# 3. Définir les nouvelles annotations.
new_scores = [
    {
        "id": "CV016",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Utilité partiellement reconnue, mais "
            "formule du top-hat incorrecte : "
            "confusion avec le gradient morphologique."
        )
    },
    {
        "id": "CV017",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Affinement des contours identifié, "
            "mais absence du maximum local dans "
            "la direction du gradient et description "
            "incorrecte de la suppression."
        )
    },
    {
        "id": "CV018",
        "version": "v1",
        "accuracy": 1,
        "completeness": 0,
        "factual_error": True,
        "justification": (
            "Précision flottante reconnue, mais "
            "absence de l'explication sur les dérivées "
            "signées et les valeurs négatives. "
            "Plage de CV_64F incorrecte. "
            "Réponse interrompue."
        )
    },
    {
        "id": "CV019",
        "version": "v1",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Identification des composantes et "
            "filtrage par aire correctement expliqués, "
            "mais API Python et exemple de code "
            "incorrects, et réponse interrompue."
        )
    },
    {
        "id": "CV020",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Lien partiel avec les anomalies de forme, "
            "mais définition incorrecte de la solidité. "
            "Rapport aire du contour sur aire de "
            "l'enveloppe convexe absent. "
            "Réponse interrompue."
        )
    }
]

# 4. Indexer les évaluations déjà enregistrées.
scores_by_key = {(item["id"], item["version"]): item for item in scores_data["evaluations"]}

assert len(scores_by_key) == len(scores_data["evaluations"])

# 5. Vérifier et intégrer les nouvelles évaluations.
for item in new_scores:

    assert item["id"] in test_references
    assert item["version"] == "v1"
    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert isinstance(item["factual_error"], bool)

    item["total_score"] = item["accuracy"] + item["completeness"]

    scores_by_key[(item["id"], item["version"])] = item

# 6. Reconstruire les résultats dans un ordre stable.
scores_data["evaluations"] = sorted(scores_by_key.values(), key=lambda item: item["id"])

# 7. Vérifier les dix évaluations attendues.
expected_ids = {f"CV{number:03d}"for number in range(11, 21)}
actual_ids = {item["id"] for item in scores_data["evaluations"]}

assert actual_ids == expected_ids
assert len(scores_data["evaluations"]) == 10

# 8. Sauvegarder le fichier mis à jour.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(scores_data, file, ensure_ascii=False, indent=2)

# 9. Calculer les résultats cumulés.
evaluations = scores_data["evaluations"]

total_score = sum(item["total_score"] for item in evaluations)
error_count = sum(item["factual_error"] for item in evaluations)
quality_percentage = 100 * total_score / (5 * len(evaluations))

print("=== Bilan des notations du jeu test ===")
print(f"Questions notées : {len(evaluations)}/40")
print(f"Score cumulé : {total_score}/50")
print(f"Qualité technique : {quality_percentage:.1f} %")
print(f"Erreurs factuelles signalées : {error_count}")

print("\nFichier :", scores_path)
print("[OK] Dix évaluations sauvegardées.")

In [ ]:
# Troisième groupe : CV021 à CV025.
selected_ids = ["CV021", "CV022", "CV023", "CV024", "CV025"]

display_questions(selected_ids)

In [ ]:
# Enregistrement des notations CV021 à CV025
import json
from pathlib import Path

# 1. Retrouver les annotations déjà sauvegardées.
scores_path = (
    results_dir / "baseline_test_v1_quality_scores_draft.json"
)

assert scores_path.exists(), (
    "Le fichier des annotations est introuvable."
)

with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

assert scores_data["split"] == "test"
assert scores_data["prompt_version"] == "v1"

# 2. Définir les cinq nouvelles notations.
new_scores = [
    {
        "id": "CV021",
        "version": "v1",
        "accuracy": 0,
        "completeness": 0,
        "factual_error": True,
        "justification": (
            "sigmaX=0 interprété comme une variance nulle, "
            "alors que l'écart-type est calculé "
            "automatiquement à partir du noyau."
        )
    },
    {
        "id": "CV022",
        "version": "v1",
        "accuracy": 2,
        "completeness": 2,
        "factual_error": True,
        "justification": (
            "Rôles de sigmaColor et sigmaSpace correctement "
            "identifiés, mais affirmations imprécises "
            "sur la détection des contours et le blanchiment."
        )
    },
    {
        "id": "CV023",
        "version": "v1",
        "accuracy": 3,
        "completeness": 1,
        "factual_error": False,
        "justification": (
            "Limites liées au bruit, au contraste et à "
            "l'éclairage correctement expliquées, mais "
            "principe du seuil global et alternative "
            "locale insuffisamment précisés."
        )
    },
    {
        "id": "CV024",
        "version": "v1",
        "accuracy": 2,
        "completeness": 2,
        "factual_error": True,
        "justification": (
            "Différence entre moyenne locale et pondération "
            "gaussienne correctement expliquée, mais "
            "généralisations sur la robustesse au bruit "
            "et exemple Python interrompu."
        )
    },
    {
        "id": "CV025",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Structures sombres reconnues, mais définition "
            "incorrecte du black-hat : la fermeture "
            "moins l'image originale est absente."
        )
    }
]

# 3. Indexer les évaluations existantes.
scores_by_key = {
    (item["id"], item["version"]): item
    for item in scores_data["evaluations"]
}

assert len(scores_by_key) == len(scores_data["evaluations"])

# 4. Vérifier et intégrer les nouvelles annotations.
for item in new_scores:
    assert item["id"] in test_references
    assert item["id"] in model_responses
    assert item["version"] == "v1"

    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert isinstance(item["factual_error"], bool)

    item["total_score"] = (
        item["accuracy"] + item["completeness"]
    )

    scores_by_key[(item["id"], item["version"])] = item

# 5. Reconstruire les annotations dans l'ordre.
scores_data["evaluations"] = sorted(
    scores_by_key.values(),
    key=lambda item: item["id"]
)

# 6. Contrôler les quinze questions attendues.
expected_ids = {
    f"CV{number:03d}"
    for number in range(11, 26)
}

actual_ids = {
    item["id"]
    for item in scores_data["evaluations"]
}

assert expected_ids.issubset(actual_ids)
assert len(scores_data["evaluations"]) >= 15

# 7. Sauvegarder les annotations actualisées.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(
        scores_data,
        file,
        ensure_ascii=False,
        indent=2
    )

# 8. Calculer le bilan cumulé.
evaluations = scores_data["evaluations"]

total_score = sum(
    item["total_score"]
    for item in evaluations
)

error_count = sum(
    item["factual_error"]
    for item in evaluations
)

quality_percentage = (
    100 * total_score / (5 * len(evaluations))
)

print("=== Bilan des notations du jeu test ===")

print(f"Questions notées : {len(evaluations)}/40")
print(
    f"Score cumulé : "
    f"{total_score}/{5 * len(evaluations)}"
)
print(f"Qualité technique : {quality_percentage:.1f} %")
print(f"Erreurs factuelles signalées : {error_count}")

print("\nFichier :", scores_path)
print("[OK] Notations sauvegardées.")


In [ ]:
# Quatrième groupe : CV026 à CV030
selected_ids = ["CV026", "CV027", "CV028", "CV029", "CV030"]

display_questions(selected_ids)

In [ ]:
# Enregistrement des notations CV026 à CV030

import json
from pathlib import Path

# 1. Charger les annotations existantes.
scores_path = (
    results_dir / "baseline_test_v1_quality_scores_draft.json"
)

assert scores_path.exists(), (
    "Le fichier des annotations est introuvable."
)

with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

assert scores_data["split"] == "test"
assert scores_data["prompt_version"] == "v1"
assert len(scores_data["evaluations"]) == 15

# 2. Préparer les cinq nouvelles notations.
new_scores = [
    {
        "id": "CV026",
        "version": "v1",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Influence de la forme de l'élément structurant "
            "reconnue, mais mécanisme du voisinage peu "
            "explicité et généralisations injustifiées "
            "sur les noyaux elliptiques et en croix."
        )
    },
    {
        "id": "CV027",
        "version": "v1",
        "accuracy": 3,
        "completeness": 2,
        "factual_error": False,
        "justification": (
            "RETR_EXTERNAL et RETR_TREE correctement "
            "distingués, avec récupération des contours "
            "et organisation hiérarchique."
        )
    },
    {
        "id": "CV028",
        "version": "v1",
        "accuracy": 2,
        "completeness": 2,
        "factual_error": True,
        "justification": (
            "Différence entre conservation et compression "
            "des points correctement expliquée, mais "
            "réduction mémoire de 50 % inventée et "
            "description imprécise des coordonnées conservées."
        )
    },
    {
        "id": "CV029",
        "version": "v1",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Connectivités 4 et 8 correctement définies, "
            "mais rôle des contacts diagonaux insuffisamment "
            "expliqué et affirmations trompeuses sur la "
            "précision et la sensibilité au bruit."
        )
    },
    {
        "id": "CV030",
        "version": "v1",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "minAreaRect correctement privilégié pour "
            "une pièce inclinée, mais confusion entre "
            "dimensions physiques et apparentes et "
            "absence de la limite liée à la perspective."
        )
    }
]

# 3. Indexer les annotations pour éviter les doublons.
scores_by_key = {
    (item["id"], item["version"]): item
    for item in scores_data["evaluations"]
}

assert len(scores_by_key) == len(scores_data["evaluations"])

# 4. Vérifier et ajouter les nouvelles notes.
for item in new_scores:
    assert item["id"] in test_references
    assert item["id"] in model_responses
    assert item["version"] == "v1"

    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert isinstance(item["factual_error"], bool)

    item["total_score"] = (
        item["accuracy"] + item["completeness"]
    )

    scores_by_key[(item["id"], item["version"])] = item

# 5. Trier les résultats par identifiant.
evaluations = sorted(
    scores_by_key.values(),
    key=lambda item: item["id"]
)

# 6. Vérifier les vingt questions attendues.
expected_ids = {
    f"CV{number:03d}"
    for number in range(11, 31)
}

actual_ids = {
    item["id"]
    for item in evaluations
}

assert len(evaluations) == 20
assert actual_ids == expected_ids

scores_data["evaluations"] = evaluations

# 7. Sauvegarder les annotations.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(
        scores_data,
        file,
        ensure_ascii=False,
        indent=2
    )

# 8. Afficher les statistiques cumulées.
total_score = sum(
    item["total_score"]
    for item in evaluations
)

error_count = sum(
    item["factual_error"]
    for item in evaluations
)

quality_percentage = (
    100 * total_score / (5 * len(evaluations))
)

print("=== Bilan des notations du jeu test ===")
print(f"Questions notées : {len(evaluations)}/40")
print(
    f"Score cumulé : "
    f"{total_score}/{5 * len(evaluations)}"
)
print(f"Qualité technique : {quality_percentage:.1f} %")
print(f"Erreurs factuelles signalées : {error_count}")

print("\nFichier :", scores_path)
print("[OK] Vingt évaluations sauvegardées.")


In [ ]:
# Cinquième groupe : CV031 à CV035
selected_ids = ["CV031", "CV032", "CV033", "CV034", "CV035"]

display_questions(selected_ids)

In [ ]:
# Enregistrement des notations CV031 à CV035

import json
from pathlib import Path

# 1. Charger le fichier provisoire existant.
scores_path = (
    results_dir / "baseline_test_v1_quality_scores_draft.json"
)

assert scores_path.exists(), (
    "Le fichier des annotations est introuvable."
)

with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

assert scores_data["split"] == "test"
assert scores_data["prompt_version"] == "v1"
assert (
    scores_data["dataset_sha256"]
    == locked_config["dataset_sha256"]
)

# 2. Définir les nouvelles notations.
new_scores = [
    {
        "id": "CV031",
        "version": "v1",
        "accuracy": 1,
        "completeness": 0,
        "factual_error": True,
        "justification": (
            "Distinction mathématique partiellement connue, "
            "mais comportement de filter2D inversé. "
            "La fonction réalise une corrélation sans "
            "retourner le noyau. Constantes inventées."
        )
    },
    {
        "id": "CV032",
        "version": "v1",
        "accuracy": 2,
        "completeness": 2,
        "factual_error": True,
        "justification": (
            "Principe d'extrapolation aux frontières "
            "correctement expliqué, mais définition "
            "de BORDER_DEFAULT incorrecte et confusion "
            "entre réflexion et réplication."
        )
    },
    {
        "id": "CV033",
        "version": "v1",
        "accuracy": 1,
        "completeness": 0,
        "factual_error": True,
        "justification": (
            "Rôle d'initialisation des marqueurs "
            "partiellement reconnu, mais représentation "
            "incorrecte et absence de l'étiquette -1 "
            "pour les frontières de Watershed."
        )
    },
    {
        "id": "CV034",
        "version": "v1",
        "accuracy": 1,
        "completeness": 0,
        "factual_error": True,
        "justification": (
            "GC_FGD identifié comme premier plan, "
            "mais GC_PR_FGD confondu avec l'arrière-plan "
            "alors qu'il indique le premier plan probable."
        )
    },
    {
        "id": "CV035",
        "version": "v1",
        "accuracy": 1,
        "completeness": 0,
        "factual_error": True,
        "justification": (
            "Notion de répétition reconnue, mais "
            "iterations=2 interprété comme deux "
            "ouvertures complètes au lieu de deux "
            "érosions suivies de deux dilatations."
        )
    }
]

# 3. Vérifier les 20 annotations précédentes.
existing_scores = scores_data["evaluations"]

expected_previous_ids = {
    f"CV{number:03d}"
    for number in range(11, 31)
}

assert len(existing_scores) == 20
assert {item["id"] for item in existing_scores} == (
    expected_previous_ids
)

# 4. Indexer les notes par question et version.
scores_by_key = {
    (item["id"], item["version"]): item
    for item in existing_scores
}

assert len(scores_by_key) == len(existing_scores)

# 5. Vérifier puis intégrer les nouvelles notes.
for item in new_scores:
    assert item["id"] in test_references
    assert item["id"] in model_responses
    assert item["version"] == "v1"

    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert isinstance(item["factual_error"], bool)

    item["total_score"] = (
        item["accuracy"] + item["completeness"]
    )

    scores_by_key[(item["id"], item["version"])] = item

# 6. Trier les 25 annotations.
evaluations = sorted(
    scores_by_key.values(),
    key=lambda item: item["id"]
)

expected_ids = {
    f"CV{number:03d}"
    for number in range(11, 36)
}

assert len(evaluations) == 25
assert {item["id"] for item in evaluations} == expected_ids

scores_data["evaluations"] = evaluations

# 7. Sauvegarder les annotations sur Google Drive.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(
        scores_data,
        file,
        ensure_ascii=False,
        indent=2
    )

# 8. Calculer le nouveau bilan.
total_score = sum(
    item["total_score"]
    for item in evaluations
)

error_count = sum(
    item["factual_error"]
    for item in evaluations
)

quality_percentage = (
    100 * total_score / (5 * len(evaluations))
)

print("=== Bilan des notations du jeu test ===")
print(f"Questions notées : {len(evaluations)}/40")
print(
    f"Score cumulé : "
    f"{total_score}/{5 * len(evaluations)}"
)
print(f"Qualité technique : {quality_percentage:.1f} %")
print(f"Erreurs factuelles signalées : {error_count}")

print("\nFichier :", scores_path)
print("[OK] Vingt-cinq évaluations sauvegardées.")

In [ ]:
# Sixième groupe : CV036 à CV040
selected_ids = ["CV036", "CV037", "CV038", "CV039", "CV040"]

display_questions(selected_ids)

In [ ]:
# Enregistrement des notations CV036 à CV040
import json
from pathlib import Path

# 1. Charger les annotations existantes.
scores_path = (
    results_dir / "baseline_test_v1_quality_scores_draft.json"
)

assert scores_path.exists(), (
    "Le fichier des annotations est introuvable."
)

with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

assert scores_data["split"] == "test"
assert scores_data["prompt_version"] == "v1"
assert (
    scores_data["dataset_sha256"]
    == locked_config["dataset_sha256"]
)

# 2. Définir les cinq nouvelles annotations.
new_scores = [
    {
        "id": "CV036",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Effets généraux sur les zones claires reconnus, "
            "mais absence des notions de minimum local "
            "et maximum local en niveaux de gris. "
            "Descriptions imprécises des intensités."
        )
    },
    {
        "id": "CV037",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Utilisation de cv2.moments et contrôle de m00 "
            "reconnus, mais formule de l'abscisse incorrecte "
            "(m11 au lieu de m10), accès invalide au "
            "dictionnaire et réponse interrompue."
        )
    },
    {
        "id": "CV038",
        "version": "v1",
        "accuracy": 2,
        "completeness": 2,
        "factual_error": True,
        "justification": (
            "Rôle d'epsilon et compromis précision-"
            "simplification correctement expliqués, "
            "mais troisième paramètre de approxPolyDP "
            "inventé : closed remplacé par min_vertices."
        )
    },
    {
        "id": "CV039",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "TM_CCOEFF_NORMED correctement associé "
            "aux scores élevés, mais TM_SQDIFF présenté "
            "comme recherchant les régions les plus "
            "différentes au lieu du minimum de différence."
        )
    },
    {
        "id": "CV040",
        "version": "v1",
        "accuracy": 2,
        "completeness": 2,
        "factual_error": True,
        "justification": (
            "Conversion BGR-HSV, bornes et masque inRange "
            "correctement expliqués, mais confusion sur "
            "le code couleur du bleu et présentation "
            "trop générale de la robustesse à l'éclairage."
        )
    }
]

# 3. Vérifier les 25 annotations précédentes.
existing_scores = scores_data["evaluations"]

expected_previous_ids = {
    f"CV{number:03d}"
    for number in range(11, 36)
}

assert len(existing_scores) == 25
assert {
    item["id"] for item in existing_scores
} == expected_previous_ids

# 4. Indexer les annotations existantes.
scores_by_key = {
    (item["id"], item["version"]): item
    for item in existing_scores
}

assert len(scores_by_key) == len(existing_scores)

# 5. Vérifier et intégrer les cinq nouvelles notes.
for item in new_scores:

    assert item["id"] in test_references
    assert item["id"] in model_responses
    assert item["version"] == "v1"

    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert isinstance(item["factual_error"], bool)

    item["total_score"] = (
        item["accuracy"] + item["completeness"]
    )

    scores_by_key[(item["id"], item["version"])] = item

# 6. Trier et vérifier les 30 évaluations.
evaluations = sorted(
    scores_by_key.values(),
    key=lambda item: item["id"]
)

expected_ids = {
    f"CV{number:03d}"
    for number in range(11, 41)
}

assert len(evaluations) == 30
assert {
    item["id"] for item in evaluations
} == expected_ids

scores_data["evaluations"] = evaluations

# 7. Sauvegarder les annotations.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(
        scores_data,
        file,
        ensure_ascii=False,
        indent=2
    )

# 8. Calculer le bilan cumulé.
total_score = sum(
    item["total_score"]
    for item in evaluations
)

error_count = sum(
    item["factual_error"]
    for item in evaluations
)

quality_percentage = (
    100 * total_score / (5 * len(evaluations))
)

print("=== Bilan des notations du jeu test ===")
print(f"Questions notées : {len(evaluations)}/40")
print(
    f"Score cumulé : "
    f"{total_score}/{5 * len(evaluations)}"
)
print(f"Qualité technique : {quality_percentage:.1f} %")
print(f"Erreurs factuelles signalées : {error_count}")

print("\nFichier :", scores_path)
print("[OK] Trente évaluations sauvegardées.")


In [ ]:
# Septième groupe : CV041 à CV045
selected_ids = ["CV041", "CV042", "CV043", "CV044", "CV045"]

display_questions(selected_ids)

In [ ]:
# Enregistrement des notations CV041 à CV045
import json
from pathlib import Path

# 1. Localiser le fichier des annotations.
results_dir = Path(
    "/content/drive/MyDrive/"
    "industrial-vision-slm-benchmark/outputs/baseline"
)

scores_path = results_dir / "baseline_test_v1_quality_scores_draft.json"
config_path = results_dir / "baseline_selected_config.json"

assert scores_path.exists()

# 2. Recharger les données et la configuration verrouillée.
with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

with config_path.open("r", encoding="utf-8") as file:
    locked_config = json.load(file)

assert scores_data["split"] == "test"
assert scores_data["prompt_version"] == "v1"
assert scores_data["dataset_sha256"] == locked_config["dataset_sha256"]

# 3. Préparer les nouvelles annotations.
new_scores = [
    {
        "id": "CV041",
        "version": "v1",
        "accuracy": 1,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Moyenne normalisée partiellement reconnue, "
            "mais normalize=False confondu avec une "
            "sortie inchangée au lieu d'une somme locale. "
            "Plage de sortie mal décrite et saturation omise."
        )
    },
    {
        "id": "CV042",
        "version": "v1",
        "accuracy": 3,
        "completeness": 2,
        "factual_error": False,
        "justification": (
            "Décomposition du filtre gaussien 2D en "
            "deux filtres 1D correctement expliquée, "
            "ainsi que les deux passes et la réduction "
            "du coût de calcul."
        )
    },
    {
        "id": "CV043",
        "version": "v1",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": False,
        "justification": (
            "Principe de distance aux frontières et "
            "utilité pour séparer des objets reconnus, "
            "mais absence de l'identification des zones "
            "internes fiables et de leur utilisation "
            "comme marqueurs Watershed."
        )
    },
    {
        "id": "CV044",
        "version": "v1",
        "accuracy": 2,
        "completeness": 1,
        "factual_error": True,
        "justification": (
            "Nécessité d'inclure entièrement l'objet "
            "correctement expliquée, mais statut de "
            "fond certain à l'extérieur non précisé. "
            "Correction du masque et itérations omises. "
            "GrabCut incorrectement assimilé à une "
            "simple descente de gradient."
        )
    },
    {
        "id": "CV045",
        "version": "v1",
        "accuracy": 1,
        "completeness": 0,
        "factual_error": True,
        "justification": (
            "Lien vague avec la recherche de motifs, "
            "mais opération HITMISS mal définie. "
            "Les valeurs 1, -1 et 0 ne sont pas "
            "correctement associées au premier plan, "
            "au fond et aux positions indifférentes."
        )
    }
]

# 4. Vérifier la présence des 30 questions précédentes.
existing_scores = scores_data["evaluations"]

expected_previous_ids = {
    f"CV{number:03d}"
    for number in range(11, 41)
}

assert len(existing_scores) == 30
assert {
    item["id"] for item in existing_scores
} == expected_previous_ids

# 5. Construire un index pour empêcher les doublons.
scores_by_key = {
    (item["id"], item["version"]): item
    for item in existing_scores
}

assert len(scores_by_key) == len(existing_scores)

# 6. Vérifier et ajouter les nouvelles notes.
for item in new_scores:

    assert item["id"] in test_references
    assert item["id"] in model_responses
    assert item["version"] == "v1"

    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert isinstance(item["factual_error"], bool)

    item["total_score"] = (
        item["accuracy"] + item["completeness"]
    )

    scores_by_key[(item["id"], item["version"])] = item

# 7. Trier les 35 évaluations.
evaluations = sorted(
    scores_by_key.values(),
    key=lambda item: item["id"]
)

expected_ids = {
    f"CV{number:03d}"
    for number in range(11, 46)
}

assert len(evaluations) == 35
assert {
    item["id"] for item in evaluations
} == expected_ids

scores_data["evaluations"] = evaluations

# 8. Sauvegarder les annotations.
with scores_path.open("w", encoding="utf-8") as file:
    json.dump(
        scores_data,
        file,
        ensure_ascii=False,
        indent=2
    )

# 9. Calculer le bilan cumulé.
total_score = sum(
    item["total_score"]
    for item in evaluations
)

error_count = sum(
    item["factual_error"]
    for item in evaluations
)

quality_percentage = (
    100 * total_score / (5 * len(evaluations))
)

print("=== Bilan des notations du jeu test ===")
print(f"Questions notées : {len(evaluations)}/40")
print(
    f"Score cumulé : "
    f"{total_score}/{5 * len(evaluations)}"
)
print(f"Qualité technique : {quality_percentage:.1f} %")
print(f"Erreurs factuelles signalées : {error_count}")

print("\nFichier :", scores_path)
print("[OK] Trente-cinq évaluations sauvegardées.")

In [ ]:
# Huitième et dernier groupe : CV046 à CV050
selected_ids = ["CV046", "CV047", "CV048", "CV049", "CV050"]

display_questions(selected_ids)

In [ ]:
# Enregistrement des notations CV046 à CV050

import json
from pathlib import Path

# 1. Localiser les fichiers persistants.
results_dir = Path(
    "/content/drive/MyDrive/"
    "industrial-vision-slm-benchmark/outputs/baseline"
)

scores_path = results_dir / "baseline_test_v1_quality_scores_draft.json"
config_path = results_dir / "baseline_selected_config.json"

# 2. Recharger les annotations et la configuration.
with scores_path.open("r", encoding="utf-8") as file:
    scores_data = json.load(file)

with config_path.open("r", encoding="utf-8") as file:
    locked_config = json.load(file)

assert scores_data["split"] == "test"
assert scores_data["prompt_version"] == "v1"
assert scores_data["dataset_sha256"] == locked_config["dataset_sha256"]

# 3. Définir les cinq dernières annotations.
new_scores = [
    {
        "id": "CV046",
        "version": "v1",
        "accuracy": 3,
        "completeness": 1,
        "factual_error": False,
        "justification": (
            "Effet de l'érosion sur les ponts étroits "
            "et risque de perte de détails correctement "
            "expliqués. Influence de la forme de "
            "l'élément structurant insuffisamment précisée."
        )
    },
    {
        "id": "CV047",
        "version": "v1",
        "accuracy": 2,
        "completeness": 2,
        "factual_error": True,
        "justification": (
            "Différence entre paramètres polaires et "
            "extrémités des segments correctement identifiée, "
            "mais opposition trompeuse entre lignes "
            "continues et lignes brisées."
        )
    },
    {
        "id": "CV048",
        "version": "v1",
        "accuracy": 3,
        "completeness": 2,
        "factual_error": False,
        "justification": (
            "minLineLength et maxLineGap correctement définis. "
            "Influence sur la longueur minimale et la "
            "continuité des segments bien expliquée."
        )
    },
    {
        "id": "CV049",
        "version": "v1",
        "accuracy": 3,
        "completeness": 1,
        "factual_error": False,
        "justification": (
            "Impossibilité de convertir directement les "
            "pixels en millimètres et nécessité d'une "
            "échelle correctement expliquées. "
            "Calibration, perspective et distorsions omises."
        )
    },
    {
        "id": "CV050",
        "version": "v1",
        "accuracy": 2,
        "completeness": 2,
        "factual_error": True,
        "justification": (
            "Effets du bruit, de l'éclairage et du "
            "désalignement correctement décrits, mais "
            "cv2.compare avec CMP_EQ présenté à tort "
            "comme une solution plus robuste."
        )
    }
]

# 4. Vérifier les 35 annotations existantes.
existing_scores = scores_data["evaluations"]

expected_previous_ids = {
    f"CV{number:03d}"
    for number in range(11, 46)
}

assert len(existing_scores) == 35
assert {
    item["id"] for item in existing_scores
} == expected_previous_ids

# 5. Construire un index pour éviter les doublons.
scores_by_key = {
    (item["id"], item["version"]): item
    for item in existing_scores
}

assert len(scores_by_key) == len(existing_scores)

# 6. Vérifier et intégrer les dernières notes.
for item in new_scores:

    assert item["id"] in test_references
    assert item["id"] in model_responses
    assert item["version"] == "v1"

    assert 0 <= item["accuracy"] <= 3
    assert 0 <= item["completeness"] <= 2
    assert isinstance(item["factual_error"], bool)

    item["total_score"] = (
        item["accuracy"] + item["completeness"]
    )

    scores_by_key[(item["id"], item["version"])] = item

# 7. Trier et vérifier les 40 annotations.
evaluations = sorted(
    scores_by_key.values(),
    key=lambda item: item["id"]
)

expected_ids = {
    f"CV{number:03d}"
    for number in range(11, 51)
}

assert len(evaluations) == 40
assert {
    item["id"] for item in evaluations
} == expected_ids

assert all(item["version"] == "v1" for item in evaluations)
assert all(
    item["total_score"]
    == item["accuracy"] + item["completeness"]
    for item in evaluations
)

scores_data["evaluations"] = evaluations

# Conserver le statut provisoire jusqu'à validation finale.
scores_data["status"] = "provisional"

# 8. Sauvegarder les données de façon atomique.
temporary_path = scores_path.with_suffix(".json.tmp")

with temporary_path.open("w", encoding="utf-8") as file:
    json.dump(
        scores_data,
        file,
        ensure_ascii=False,
        indent=2
    )

temporary_path.replace(scores_path)

# 9. Calculer les statistiques de qualité.
total_score = sum(
    item["total_score"]
    for item in evaluations
)

error_count = sum(
    item["factual_error"]
    for item in evaluations
)

quality_percentage = (
    100 * total_score / (5 * len(evaluations))
)

error_percentage = (
    100 * error_count / len(evaluations)
)

# 10. Afficher le bilan complet.
print("=== Bilan complet des notations du jeu test ===")

print(f"Questions notées : {len(evaluations)}/40")
print(f"Score cumulé : {total_score}/200")
print(f"Qualité technique : {quality_percentage:.1f} %")

print(
    f"Réponses avec erreurs factuelles : "
    f"{error_count}/40 ({error_percentage:.1f} %)"
)

print("Statut : provisoire, en attente de validation finale")
print("\nFichier :", scores_path)
print("[OK] Quarante évaluations sauvegardées.")

### *6.5. Validation et synthèse définitive de la baseline*

**Objectif :** finaliser le benchmark initial de Qwen3-1.7B sur le jeu de test verrouillé.

**Méthode :**
1. Recharger les réponses, les annotations et les métriques.
2. Vérifier la cohérence des identifiants, scores et configurations.
3. Calculer la qualité globale et par domaine.
4. Comparer les réponses ayant atteint la limite de génération aux autres.
5. Sauvegarder les évaluations définitives et leur synthèse.

**Résultats attendus :**
- `baseline_test_v1_quality_scores_final.json`
- `baseline_test_v1_quality_summary.json`
- Une mesure de référence reproductible pour les futures expériences RAG et QLoRA.

**Point de vigilance :** le fichier provisoire est conservé. Aucun score, prompt, paramètre de génération ou exemple du jeu de test n'est modifié pendant la finalisation.

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone

# 1. Chemins des fichiers persistants
results_dir = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/baseline")
draft_path = results_dir / "baseline_test_v1_quality_scores_draft.json"
final_path = results_dir / "baseline_test_v1_quality_scores_final.json"
quality_summary_path = results_dir / "baseline_test_v1_quality_summary.json"
results_path = results_dir / "baseline_test_v1_results.json"
performance_path = results_dir / "baseline_test_v1_summary.json"
config_path = results_dir / "baseline_selected_config.json"

# 2. Recharge des fichiers
def load_json(path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

scores_data = load_json(draft_path)
benchmark_data = load_json(results_path)
performance_data = load_json(performance_path)
locked_config = load_json(config_path)

# 3. Vérif de la config expérimentale
assert scores_data["split"] == "test"
assert scores_data["prompt_version"] == "v1"
assert scores_data["status"] == "provisional"

assert locked_config["prompt_version"] == "v1"
assert locked_config["model_id"] == "Qwen/Qwen3-1.7B"

assert scores_data["dataset_sha256"] == locked_config["dataset_sha256"]

assert benchmark_data['split'] == 'test'

for key, value in benchmark_data["configuration"].items():
    assert locked_config[key] == value

print("[OK] Configuration expérimentale cohérente.")

# 4. Indexation des annotations et réponses
evaluations = scores_data["evaluations"]
records = benchmark_data["results"]

scores_by_id = {item["id"]: item for item in evaluations}

records_by_id = {item["id"]: item for item in records}

expected_ids = {f"CV{number:03d}" for number in range(11,51)}

assert len(evaluations) == 40
assert len(records) == 40
assert len(scores_by_id) == 40
assert len(records_by_id) == 40

assert set(scores_by_id) == expected_ids
assert set(records_by_id) == expected_ids
assert set(benchmark_data["expected_ids"]) == expected_ids

# 5. Vérif de chaque notation
for evaluation in evaluations:

    assert evaluation["version"] == "v1"
    assert 0 <= evaluation["accuracy"] <= 3
    assert 0 <= evaluation["completeness"] <= 2

    assert evaluation["total_score"] ==  evaluation["accuracy"] + evaluation["completeness"]

    assert isinstance(evaluation["factual_error"], bool)
    assert evaluation["justification"].strip()

# 6. Vérif des mesures de génération
for record in records:
    assert record["response"].strip()
    assert record["generated_tokens"] > 0

    assert record["generation_limit_reached"] == (record["generated_tokens"] >= locked_config["max_new_tokens"])

performance_stats = performance_data["global_statistics"]
assert performance_stats["question_count"] == 40
assert performance_stats["total_generated_tokens"] == sum(record["generated_tokens"] for record in records)
assert performance_stats["generation_limit_count"] == sum(record["generation_limit_reached"] for record in records)

print("[OK] Quarante annotations & générations vérifiées.")


# 7. Fonction de calcul des statistiques de qualité.
def compute_quality_statistics(items):
    """Calculer les métriques de qualité d'un groupe d'annotations."""

    question_count = len(items)

    accuracy_total = sum(item["accuracy"] for item in items)
    completeness_total = sum(item["completeness"] for item in items)
    score_total = sum(item["total_score"] for item in items)
    error_count = sum(item["factual_error"] for item in items)

    return {
        "question_count": question_count,
        "accuracy_total": accuracy_total,
        "completeness_total": completeness_total,
        "total_score": score_total,
        "max_score": 5 * question_count,
        "mean_score_on_5": score_total / question_count,
        "quality_percentage":  100 * score_total / (5 * question_count),
        "factual_error_count": error_count,
        "factual_error_percentage": 100 * error_count / question_count
    }

# 8. Métriques globales
global_quality = compute_quality_statistics(evaluations)

assert global_quality["total_score"] == 113
assert global_quality["factual_error_count"] == 31

# 9. Regroupement des annotations par domaine
quality_groups = defaultdict(list)

for evaluation in evaluations:
    topic = records_by_id[evaluation["id"]]["topic"]
    quality_groups[topic].append(evaluation)

assert len(quality_groups) == 5

quality_by_topic = {}

for topic, items in sorted(quality_groups.items()):
    assert len(items) == 8
    quality_by_topic[topic] = compute_quality_statistics(items)

# 10. Comparer les réponses selon la limite de génération.
limited_groups = {
    "limit_reached": [],
    "limit_not_reached": []
}

for evaluation in evaluations:
    result = records_by_id[evaluation["id"]]
    group_name = "limit_reached" if result["generation_limit_reached"] else "limit_not_reached"
    limited_groups[group_name].append(evaluation)

quality_by_generation_limit = {
    group_name: compute_quality_statistics(items)
    for group_name, items in limited_groups.items()
}

assert len(limited_groups["limit_reached"]) == 9
assert len(limited_groups["limit_not_reached"]) == 31

# 11. Affichage des stats globales
print("\n=== Qualité globale de la baseline ===")

print(f"Questions notées : {global_quality['question_count']}/40")
print(f"Score : {global_quality['total_score']}/{global_quality['max_score']}")
print(f"Qualité technique : {global_quality['quality_percentage']:.1f} %")
print(
    f"Réponses avec erreurs factuelles : "
    f"{global_quality['factual_error_count']}/40 "
    f"({global_quality['factual_error_percentage']:.1f} %)"
)

# 12. Affichage des résultats par domaine
print("\n=== Qualité par domaine ===")

for topic, stats in quality_by_topic.items():
    print(
        f"{topic}: "
        f"{stats['total_score']}/40 "
        f"({stats['quality_percentage']:.1f} %), "
        f"{stats['factual_error_count']} erreurs"
    )

# 13. Affichage des résultats selon la limite
print("\n=== Influence de la limite de génération ===")

for group_name, stats in quality_by_generation_limit.items():
    print(
        f"{group_name}: "
        f"{stats['total_score']}/{stats['max_score']} "
        f"({stats['quality_percentage']:.1f} %), "
        f"{stats['factual_error_count']} erreurs"
    )

# 14. Construction des fichiers définitifs
finalized_at = datetime.now(timezone.utc).isoformat()
final_scores_data = {
    **scores_data,
    "status": "final",
    "finalized_at_utc": finalized_at
}

quality_summary_data = {
    "model_id": locked_config["model_id"],
    "prompt_version": locked_config["prompt_version"],
    "split": "test",
    "dataset_sha256": locked_config["dataset_sha256"],
    "status": "final",
    "finalized_at_utc": finalized_at,
    "evaluation_protocol": {
        "accuracy_max": 3,
        "completeness_max": 2,
        "max_score_per_question": 5,
        "factual_error_is_independent": True
    },
    "source_files": {
        "responses": results_path.name,
        "performance": performance_path.name,
        "annotations": final_path.name
    },
    "global_quality": global_quality,
    "quality_by_topic": quality_by_topic,
    "quality_by_generation_limit": (
        quality_by_generation_limit
    ),
    "performance": {
        "question_count": performance_stats["question_count"],
        "mean_latency_seconds": (
            performance_stats["mean_latency_seconds"]
        ),
        "global_throughput": (
            performance_stats["global_throughput"]
        ),
        "max_gpu_memory_gib": (
            performance_stats["max_gpu_memory_gib"]
        ),
        "generation_limit_count": (
            performance_stats["generation_limit_count"]
        )
    }
}

# 15. Sauvegarde indépendante du fichier provisoire
def save_json_atomic(file_path, data):
    """Écrire un JSON via un fichier temporaire."""

    temporary_path = file_path.with_suffix(".json.tmp")

    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

    temporary_path.replace(file_path)

save_json_atomic(final_path, final_scores_data)
save_json_atomic(quality_summary_path, quality_summary_data)

# 16. Recharge des fichiers pour vérifier les sauvegardes
saved_final = load_json(final_path)
saved_summary = load_json(quality_summary_path)

assert saved_final["status"] == "final"
assert len(saved_final["evaluations"]) == 40

assert saved_summary["global_quality"]["total_score"] == 113
assert saved_summary["global_quality"]["factual_error_count"] == 31

print("\n=== Fichiers définitifs ===")
print("Annotations :", final_path.name)
print("Synthèse :", quality_summary_path.name)

print("\n[OK] Benchmark initial validé et finalisé.")


---

## Conclusion — Benchmark du modèle initial

### Objectif atteint

Cette première expérience établit une baseline reproductible pour
Qwen3-1.7B sur des questions techniques de traitement d'images,
d'OpenCV et de vision industrielle.

### Configuration expérimentale

- **Modèle :** Qwen/Qwen3-1.7B.
- **Précision :** FP16.
- **Matériel :** GPU NVIDIA Tesla T4.
- **Configuration :** prompt V1 sélectionné sur le jeu de développement.
- **Génération :** déterministe, sans mode thinking.
- **Limite :** 512 nouveaux tokens par réponse.
- **Dataset :** 50 questions, réparties en 10 questions de développement et 40 questions de test.
- **Domaines :** filtrage, segmentation, morphologie, contours et inspection industrielle.

### Résultats sur le jeu de test

| Métrique | Résultat |
|---|---:|
| Questions évaluées | 40/40 |
| Qualité technique | 113/200 (56,5 %) |
| Réponses avec erreur factuelle signalée | 31/40 (77,5 %) |
| Générations atteignant la limite | 9/40 (22,5 %) |
| Latence moyenne | 16,932 s |
| Débit global | 24,10 tokens/s |
| Pic mémoire GPU PyTorch | 3,31 Gio |

### Qualité par domaine

| Domaine | Score |
|---|---:|
| Contours | 27/40 |
| Filtrage | 26/40 |
| Inspection industrielle | 25/40 |
| Segmentation | 18/40 |
| Morphologie | 17/40 |

### Principales observations

Le modèle parvient à expliquer plusieurs notions générales de
traitement d'images, mais rencontre des difficultés sur certaines
définitions mathématiques, signatures de fonctions OpenCV et
distinctions entre opérations proches.

Les domaines de la morphologie et de la segmentation présentent
les scores les plus faibles dans ce jeu de test.

La longueur et la fluidité d'une réponse ne garantissent pas
son exactitude scientifique.

### Limites de l'évaluation

- Le jeu de test contient seulement 40 questions.
- Les scores de qualité reposent sur une annotation manuelle.
- Le taux d'erreurs factuelles indique la présence d'au moins
  une erreur signalée, pas nécessairement une réponse entièrement fausse.
- Les mesures de performance dépendent du matériel et des
  conditions d'exécution.

### Perspectives

Cette baseline servira de référence pour comparer les prochaines
approches du projet :

1. RAG avec recherche documentaire spécialisée.
2. Adaptation du modèle avec LoRA/QLoRA.
3. Comparaison de la qualité, de la latence et de la mémoire.
4. Analyse des erreurs et des compromis entre les méthodes.

La configuration et le jeu de test resteront inchangés afin
de préserver la comparabilité des expériences.